**Running this on Google Colab.** The first code cell fetches this entry's files and installs the packages it was run with, pinned for Colab's Python version. The results in the registry come from the entry's own environment ([requirements.lock.txt](requirements.lock.txt), Python 3.13); a different Python or platform can change the last digits of some numbers.

In [ ]:
# On Google Colab: fetch this entry's files and install its pinned packages. Elsewhere this cell does nothing.
import sys
if "google.colab" in sys.modules:
    import os, subprocess
    if not os.path.isdir("/content/encore"):
        subprocess.run(["git", "clone", "-q", "--depth", "1", "--branch", "colab", "--filter=blob:none", "--sparse",
                        "https://github.com/encorescience/encore-registry", "/content/encore"], check=True)
        subprocess.run(["git", "-C", "/content/encore", "sparse-checkout", "set", "entries/tingley-2019"], check=True)
    os.chdir("/content/encore/entries/tingley-2019")
    sys.path.insert(0, os.getcwd())
    version = f"{sys.version_info.major}.{sys.version_info.minor}"
    lock = f"requirements.colab-{version}.txt"
    if not os.path.exists(lock):
        print(f"No lock for Python {version}: installing the entry's unpinned requirements (requirements.in).")
        lock = "requirements.in"
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "uv"], check=True)
    subprocess.run(["uv", "pip", "install", "--system", "-q", "-r", lock], check=True)
    print("Installed. If Colab asks to restart the session, restart it and run the notebook from the top again.")


# Reproducing Tingley and Buzsaki (2019) from DANDI:000218: spikes and events

Tingley D. and Buzsaki G. (2019). Routing of hippocampal ripples to subcortical structures via the lateral septum. Neuron 105, 138-149. doi:10.1016/j.neuron.2019.10.012.

Dataset: DANDI:000218, version 0.261110.1911 (doi:10.48324/dandi.000218/0.261110.1911), streamed from the archive.

Version. The paper cites no version of the dandiset (it predates it). The most recent published version, 0.261110.1911, is pinned: its files were re-converted with trial labels sorted with their rows, the lab's ripple and LS HFO detections, sleep states for every session with them, units with shank and region, and electrode flags, all of which the claims use; the earlier versions (0.220131.1608, 0.250624.0439) lack the event tables and sleep states, so they match the claims file's description less well. The LFP is the earlier version's data, stored again in channel order (release notes).

Claims file: `claims/tingley-2019.yaml`, written from the accepted manuscript in PubMed Central (PMC6952543) with its supplement. The paper's full text and supplement were read here.

This notebook tests the claims that use spikes, the lab's event tables, sleep states and trials. The LFP analyses are in `lfp_tingley_2019.ipynb` and the Figure 7E regressions in `fig7e_tingley_2019.ipynb`.

Claims and where they are tested (all notebooks of this entry):

| Claim | Where tested |
|---|---|
| dataset-composition | main notebook (counts); LFP notebook (LS neurons with more than 50 spikes) |
| fig1c-hfo-each-animal, fig1c-post-burst-suppression, fig1c-event-count, figs1-participation | main notebook |
| fig1d-phase-locking | LFP notebook; own-shank notebook (shanks outside the detection channel's storage block) |
| fig1f-shorter-than-ripples, fig1g-slower-than-ripples, fig1h-no-amplitude-frequency-correlation | main notebook |
| fig2-confined-to-ls | LFP notebook |
| fig2ef-intermediate-ls | blocked: metadata_insufficient (no relative depth of each recording within LS) |
| fig3-travelling-waves | blocked: metadata_insufficient (no electrode positions) |
| fig4b-pre-post-unchanged, fig4b-hfo-above-behav, fig4-sessions-few-change, fig4c-temporal-bias, fig4d-compression | main notebook |
| figs3d-hpc-compression-post, figs2-post-rate-increase, figs2c-no-ls-reactivation | main notebook |
| fig5b-nrem-and-rates, fig5e-state-coupling, fig5f-ripple-triggered-firing | main notebook |
| fig5d-lag | main notebook (deposited peak times); LFP notebook (maximum-NSS times) |
| fig5f-no-phase-coherence, ls-spikes-not-locked-to-ca1-ripples | LFP notebook |
| fig6-optogenetic-hfos | blocked: data_missing (the two optogenetic rats are not in the dandiset) |
| fig7e-ca1-spikes, fig7e-state, fig7e-duration | Figure 7E notebook |
| fig7e-ca1-depth | blocked: metadata_insufficient (no site positions or per-site waveforms) |
| fig7e-replay, fig7e-seqnmf, figs5-seqnmf-variance | blocked: other (not attempted within the run's time) |
| fig7e-content-independent, fig7e-coupled-only | blocked: other (the content features, fig7e-replay and fig7e-seqnmf, were not attempted) |
| discussion-medial-septum | main notebook (part b); LFP notebook (part a) |
| discussion-cingulate-down-state | main notebook (reading a) |
| fig8-model | blocked: other (the model's data are elsewhere and it was not attempted within the run's time) |

## Setup
Every file of the pinned version is streamed; from each, the spikes, units, event tables, sleep states,
trials and electrode flags are read into memory. The LFP is not read in this notebook.

In [ ]:
import zlib, time, resource, collections, warnings
import numpy as np, pandas as pd
from scipy import stats
import matplotlib.pyplot as plt
import tingley2019 as T
warnings.filterwarnings("ignore", category=RuntimeWarning)
pd.set_option("display.width", 200); pd.set_option("display.max_columns", 30); pd.set_option("display.max_rows", 200)
KEY = "tingley-2019"
def seed_of(claim): return zlib.crc32(f"{KEY}/{claim}".encode())
t0 = time.time()
A = T.list_assets()
S = T.load_all(A, workers=8)
print(f"{len(S)} files of dandiset {T.DANDISET} version {T.VERSION} loaded in {time.time()-t0:.0f} s; "
      f"peak memory {resource.getrusage(resource.RUSAGE_SELF).ru_maxrss/1e6:.2f} GB")

## Data check
What was loaded, by subject and session; the ranges and units of every variable used; recordings deposited
twice; and whether the spikes, the event tables, the sleep states and the trials share one clock.

In [ ]:
def ls_ev(s, fallback=False):
    """The lab's main LS HFO detection (LSRipples); with fallback, the earlier ls_ripples, then ripples_ls."""
    for k in (["LSRipples", "ls_ripples", "ripples_ls"] if fallback else ["LSRipples"]):
        if k in s.events and len(s.events[k]):
            return s.events[k]
    return None
def ca1_ev(s, fallback=False):
    """The lab's CA1 ripples (CA1Ripples); with fallback, the hippocampal ripplesALL."""
    for k in (["CA1Ripples", "ripplesALL"] if fallback else ["CA1Ripples"]):
        if k in s.events and len(s.events[k]):
            return s.events[k]
    return None
HPC = list(T.HPC_REGIONS)
rows = []
for s in S:
    reg = collections.Counter(s.unit_region)
    sl = {} if s.sleep is None else (s.sleep.stop_time - s.sleep.start_time).groupby(s.sleep.label).sum().to_dict()
    rows.append(dict(subject=s.subject, session=s.session[:30], hours=round(s.duration/3600, 2), lfp_channels=s.lfp_shape[1],
                     ls_units=reg.get("ls", 0), hpc_units=reg.get("hpc", 0), ca1_units=reg.get("ca1", 0), ca3_units=reg.get("ca3", 0),
                     other_units=sum(v for k, v in reg.items() if k not in ("ls", "hpc", "ca1", "ca3")),
                     other_regions=",".join(sorted(k if k else "''" for k in reg if k not in ("ls", "hpc", "ca1", "ca3"))),
                     tables=",".join(f"{k}:{len(v)}" for k, v in sorted(s.events.items())),
                     nrem_h=round(sl.get("NREM", 0)/3600, 2), wake_h=round(sl.get("WAKE", 0)/3600, 2), rem_h=round(sl.get("REM", 0)/3600, 2),
                     trials=0 if s.trials is None else len(s.trials),
                     ca3_flag=bool(s.electrodes.ca3_channel.any()), ca1_flag=bool(s.electrodes.ca1_channel.any())))
INV = pd.DataFrame(rows)
print(INV.to_string())
print("\nBy subject:")
print(INV.groupby("subject")[["ls_units", "hpc_units", "ca3_units", "other_units", "trials"]].sum()
      .join(INV.groupby("subject").size().rename("files")).to_string())
print("\nUnit regions over all files:", dict(collections.Counter(r for s in S for r in s.unit_region)))
print("Event tables (files holding each):", dict(collections.Counter(k for s in S for k in s.events)))

In [ ]:
# Ranges and units of the variables used.
for k in ["LSRipples", "CA1Ripples", "ripplesALL", "ls_ripples", "ripples_ls"]:
    d = pd.concat([s.events[k] for s in S if k in s.events])
    dur = (d.stop_time - d.start_time) * 1000
    inside = np.mean([((s.events[k].start_time >= 0) & (s.events[k].stop_time <= s.duration + 1)).mean() for s in S if k in s.events])
    pk = ((d.peak_time >= d.start_time) & (d.peak_time <= d.stop_time)).mean()
    fr = f"peak_frequency {d.peak_frequency.min():.1f} to {d.peak_frequency.max():.1f} (median {d.peak_frequency.median():.1f}, read as Hz)" if "peak_frequency" in d else "no peak_frequency"
    print(f"{k}: {len(d)} events; duration {dur.min():.1f} to {dur.max():.1f} ms (median {dur.median():.1f}); {fr}; "
          f"peak inside start-stop {pk:.3f}; events inside the recording {inside:.3f}")
allsp = [(s.spikes[i].min(), s.spikes[i].max(), s.duration) for s in S for i in range(len(s.spikes)) if len(s.spikes[i])]
print(f"Spike times: {sum(len(x) for s in S for x in s.spikes)} spikes in {sum(len(s.spikes) for s in S)} units; "
      f"min {min(a for a,_,_ in allsp):.4f} s; units with a spike after the LFP's end: {sum(b > d + 1 for _, b, d in allsp)}")
print("Sorted spike times in every unit:", all(np.all(np.diff(x) >= 0) for s in S for x in s.spikes))
sl = pd.concat([s.sleep.assign(dur=s.duration) for s in S if s.sleep is not None])
print(f"Sleep states: labels {sorted(sl.label.unique())}; bouts {len(sl)}; stop beyond the recording: {(sl.stop_time > sl.dur + 1).sum()}; "
      f"whole-second times: {np.mean(np.isclose(sl.start_time % 1, 0)):.3f}")
tr = pd.concat([s.trials.assign(dur=s.duration) for s in S if s.has_trials])
print(f"Trials: {len(tr)} in {sum(s.has_trials for s in S)} files; length {(tr.stop_time-tr.start_time).min():.2f} to "
      f"{(tr.stop_time-tr.start_time).max():.1f} s (median {(tr.stop_time-tr.start_time).median():.1f}); beyond the recording: {(tr.stop_time > tr.dur + 1).sum()}")

In [ ]:
# Recordings deposited twice: identical spike trains (unit spike counts and first spikes) or identical event tables.
fp_sp, fp_ev, dup = {}, {}, []
for i, s in enumerate(S):
    if len(s.spikes):
        key = tuple(sorted((len(x), round(float(x[0]), 4) if len(x) else 0) for x in s.spikes))
        if key in fp_sp: dup.append(("spikes", S[fp_sp[key]].session, s.session))
        fp_sp[key] = i
    for k, e in s.events.items():
        key = (len(e), tuple(np.round(e.start_time.to_numpy()[:20], 4)))
        if len(e) > 5 and key in fp_ev and fp_ev[key][0] != i:
            dup.append((f"events {k} / {fp_ev[key][1]}", S[fp_ev[key][0]].session, s.session))
        fp_ev[key] = (i, k)
# also: same subject, same LFP length
lens = collections.defaultdict(list)
for s in S: lens[(s.subject, s.lfp_shape[0])].append(s.session)
dup += [("same LFP length", *v) for v in lens.values() if len(v) > 1]
print("Duplicate candidates:", dup if dup else "none")
# Within one file, two event tables holding the same events (an LS table copied into another) are reported too.
same = []
for s in S:
    ks = sorted(s.events)
    for a in range(len(ks)):
        for b in range(a + 1, len(ks)):
            ea, eb = s.events[ks[a]].peak_time.to_numpy(), s.events[ks[b]].peak_time.to_numpy()
            if len(ea) and len(eb):
                j = np.clip(np.searchsorted(eb, ea), 1, len(eb) - 1)
                near = np.minimum(np.abs(eb[j] - ea), np.abs(eb[j - 1] - ea)) < 0.002
                if near.mean() > 0.9 and abs(len(ea) - len(eb)) < 0.1 * len(ea):
                    same.append((s.session[:25], ks[a], ks[b], len(ea), len(eb)))
print("Tables within a file with the same events:", same if same else "none")

In [ ]:
# One clock. (1) LS spikes around LS HFO peak times and hippocampal spikes around CA1 ripple peak times: the rate
# within +-10 ms over the rate 100-200 ms away, with the true times and with the times shifted by 1 s.
# (2) Trials against sleep scoring: the fraction of trial time scored WAKE, with the true states and with
# the states shifted by +-1800 s. A shared clock gives a ratio far above 1 at the true times only, and trials in WAKE.
def peri_ratio(s, units, times):
    c = sum(T.triggered_counts(s.spikes[u], times, 0.2, 0.01).sum(0) for u in units)
    lag = np.arange(-20, 21)
    return c[np.abs(lag) <= 1].mean() / max(c[(np.abs(lag) >= 10)].mean(), 1e-9)
clk = []
for s in S:
    ls = s.units_in(["ls"]); hp = s.units_in(HPC)
    e1, e2 = ls_ev(s), ca1_ev(s)
    r = dict(session=s.session[:25], subject=s.subject)
    if e1 is not None and len(ls):
        t = e1.peak_time.to_numpy(); r["ls_hfo"] = peri_ratio(s, ls, t); r["ls_hfo_shift1s"] = peri_ratio(s, ls, t + 1.0)
    if e2 is not None and len(hp):
        t = e2.peak_time.to_numpy(); r["ca1_rip"] = peri_ratio(s, hp, t); r["ca1_rip_shift1s"] = peri_ratio(s, hp, t + 1.0)
    if s.has_trials and s.sleep is not None:
        wk = s.state_intervals("WAKE"); tiv = s.trials[["start_time", "stop_time"]].to_numpy()
        r["trial_in_wake"] = T.total(T.intersect(tiv, wk)) / T.total(tiv)
        r["trial_in_wake_shift"] = np.mean([T.total(T.intersect(tiv, wk + d)) / T.total(tiv) for d in (-1800, 1800)])
    clk.append(r)
CLK = pd.DataFrame(clk)
print(CLK.describe().loc[["count", "min", "50%", "max"]].round(3).to_string())
print("Files where the LS HFO ratio is below 1.5:", CLK[CLK.ls_hfo < 1.5].session.tolist())
print("Files where the CA1 ripple ratio is below 1.5:", CLK[CLK.ca1_rip < 1.5].session.tolist())
print("Files where less than half the trial time is scored WAKE:", CLK[CLK.trial_in_wake < 0.5][["session", "trial_in_wake"]].values.tolist())

In [ ]:
# Every column of the electrode and units tables, in one file per animal (verifier round 1: is there a site position or a per-site
# waveform anywhere?). Shapes are given; position columns, if any, are summarized.
for sub in sorted({s.subject for s in S}):
    s = next(x for x in S if x.subject == sub and len(x.spikes))
    f = T.open_nwb(s.url)
    el = f["general/extracellular_ephys/electrodes"]; un = f["units"]
    print(f"{sub} ({s.session[:30]}):")
    print("  electrodes:", {k: el[k].shape for k in el.keys()})
    for k in ("x", "y", "z", "rel_x", "rel_y", "rel_z", "location"):
        if k in el:
            v = el[k][:]
            print(f"    {k}: {len(np.unique(v))} distinct values, e.g. {[T._s(x) if isinstance(x, bytes) else x for x in np.unique(v)[:5]]}")
    print("  units:", {k: un[k].shape for k in un.keys()})
    print("  extracellular_ephys groups:", [k for k in f["general/extracellular_ephys"].keys() if k != "electrodes"][:12])
    f.close()

## dataset-composition
> LS was recorded in 6 rats with 1989 LS neurons in the phase locking analysis; 62 recordings had a hippocampal
> population with LS, 37 with CA1 and 25 with CA3; LS HFO rates come from 89 recordings and CA1 ripple rates from 53;
> recordings held 8 to 74 LS neurons (mean 29.6).

Reading (a) of the LS neuron count (more than 50 spikes in high-power epochs) needs the LFP and is computed in the
second notebook (`lfp_tingley_2019.ipynb`); here reading (b), all LS units. CA1 against CA3 is read two ways, since
most hippocampal units are labelled only `hpc`: (i) by the units' label (`ca3` is CA3, `hpc` or `ca1` is CA1),
(ii) by the electrodes' `ca3_channel` flag (a recording with a CA3 channel marked is CA3). Rate recordings are those
with the event table and scored sleep (NREM or WAKE).

In [ ]:
comp = {}
comp["animals with LS units"] = INV[INV.ls_units > 0].subject.nunique()
comp["LS units, all (reading b)"] = int(INV.ls_units.sum())
both = INV[(INV.ls_units > 0) & (INV.hpc_units + INV.ca1_units + INV.ca3_units > 0)]
comp["recordings with LS and hippocampal units"] = len(both)
comp["  CA3 by unit label (i)"] = int((both.ca3_units > 0).sum()); comp["  CA1 by unit label (i)"] = int((both.ca3_units == 0).sum())
comp["  CA3 by ca3_channel flag (ii)"] = int(both.ca3_flag.sum()); comp["  CA1 by flag (ii)"] = int((~both.ca3_flag).sum())
has_sleep = INV.nrem_h + INV.wake_h > 0
comp["recordings with LSRipples and sleep states"] = int(sum(ls_ev(s) is not None and s.sleep is not None for s in S))
comp["  same, any LS table"] = int(sum(ls_ev(s, True) is not None and s.sleep is not None for s in S))
comp["recordings with CA1Ripples and sleep states"] = int(sum(ca1_ev(s) is not None and s.sleep is not None for s in S))
comp["  same, CA1Ripples or ripplesALL"] = int(sum(ca1_ev(s, True) is not None and s.sleep is not None for s in S))
n_ls = INV.ls_units[INV.ls_units > 0]
comp["LS units per recording, min"] = int(n_ls.min()); comp["LS units per recording, max"] = int(n_ls.max())
comp["LS units per recording, mean"] = round(n_ls.mean(), 1); comp["LS units per recording, SD"] = round(n_ls.std(), 1)
comp["recordings with LS units"] = len(n_ls)
for k, v in comp.items(): print(f"{k:45s} {v}")
print("\nLS units and recordings by animal:")
print(INV[INV.ls_units > 0].groupby("subject").ls_units.agg(["count", "sum", "min", "max", "mean"]).round(1).to_string())

**Result: dataset-composition, counts from spikes and tables.** Six animals have LS units (DT1, DT2, DT5, DT7, DT8, DT9), as in the paper. All LS units number 2551 (reading b; outside 1989 +- 198.9); reading (a) is in the LFP notebook. 48 recordings hold LS and hippocampal units: by the units' label 38 CA1 (within 37 +- 3.7) and 10 CA3 (outside 25 +- 2.5); by the `ca3_channel` flag 29 and 19, both outside. LS HFO rates come from 77 recordings with LSRipples and sleep states (80 with any LS table; 89 +- 8.9 needs at least 80.1), CA1 ripple rates from 47 with CA1Ripples (outside 53 +- 5.3) or 55 with ripplesALL as well (within). Recordings hold 8 to 74 LS units, mean 31.1 +- 15.5 (paper 8 to 74, 29.6 +- 15.5). Two of the counts within tolerance rest on choices the paper does not make: 38 CA1 recordings counts units labelled `hpc` as CA1 (29 by the flag), and 55 CA1 ripple rate recordings counts ripplesALL where CA1Ripples is missing (47 with CA1Ripples alone). The outcome is decided with reading (a) of the LS neuron count, in the LFP notebook.

## fig1c-event-count
> Across 6 animals, 29,955 LS HFO events were analysed.

Detection reading (a): the deposited events are counted. Recording readings: (a) all recordings with LS electrodes,
(b) recordings with LS units. The main table is LSRipples; the earlier tables are shown as variants.

In [ ]:
ev_rows = []
for s in S:
    has_ls_el = bool((s.electrodes.region == "ls").any())
    e, e_any = ls_ev(s), ls_ev(s, True)
    ev_rows.append(dict(subject=s.subject, ls_electrodes=has_ls_el, ls_units=int((s.unit_region == "ls").sum()),
                        LSRipples=0 if e is None else len(e), any_table=0 if e_any is None else len(e_any)))
EV = pd.DataFrame(ev_rows)
cnt_a = EV[EV.ls_electrodes].LSRipples.sum(); cnt_b = EV[EV.ls_units > 0].LSRipples.sum()
print(f"LSRipples, recordings with LS electrodes (a): {cnt_a} events in {EV[EV.ls_electrodes & (EV.LSRipples>0)].subject.nunique()} animals, "
      f"{(EV.ls_electrodes & (EV.LSRipples>0)).sum()} recordings")
print(f"LSRipples, recordings with LS units (b): {cnt_b}")
print(f"Variant, LSRipples else ls_ripples else ripples_ls, (a): {EV[EV.ls_electrodes].any_table.sum()}, (b): {EV[EV.ls_units>0].any_table.sum()}")
print(f"Paper 29955, tolerance 2995.5: (a) {abs(cnt_a-29955)<=2995.5}, (b) {abs(cnt_b-29955)<=2995.5}")
print(EV.groupby("subject")[["LSRipples", "any_table"]].sum().to_string())

**Result: fig1c-event-count reproduced.** LSRipples holds 27014 events in the 77 recordings with LS electrodes and an LS table, in 6 animals (reading a), within 29955 +- 2995.5; counting only recordings with LS units (reading b) gives 25690, outside. Using the earlier LS tables where LSRipples is missing gives 28978 and 27654.

## fig1f-shorter-than-ripples, fig1g-slower-than-ripples
> LS HFOs are shorter than CA1 ripples from the same animals (27 +- 9 against 31 +- 14 ms; rank-sum P < 0.01).
> LS HFOs are slower in frequency than CA1 ripples (147 +- 5.7 against 159 +- 8 Hz; rank-sum P < 0.01).

Durations are stop minus start of the deposited events; frequency is the deposited `peak_frequency` (bz_FindRipples'
instantaneous frequency at the peak, in Hz). Readings: (1) events from recordings with both tables, (2) all
events of the animals with both.

In [ ]:
def pooled(tab_fn, sessions):
    d = [tab_fn(s).assign(subject=s.subject, session=s.session) for s in sessions if tab_fn(s) is not None]
    return pd.concat(d) if d else None
both_rec = [s for s in S if ls_ev(s) is not None and ca1_ev(s) is not None]
animals_both = sorted({s.subject for s in both_rec})
readings = {"(1) recordings with both": (both_rec, both_rec),
            "(2) all events of animals with both": ([s for s in S if s.subject in animals_both],) * 2}
DUR = {}
for name, (sl, sc) in readings.items():
    L, C = pooled(ls_ev, sl), pooled(ca1_ev, sc)
    Ld, Cd = (L.stop_time - L.start_time) * 1000, (C.stop_time - C.start_time) * 1000
    pd_ = stats.mannwhitneyu(Ld, Cd); pf = stats.mannwhitneyu(L.peak_frequency, C.peak_frequency)
    DUR[name] = (L, C)
    print(f"{name}: {len(L)} LS HFOs, {len(C)} CA1 ripples, {L.subject.nunique()} / {C.subject.nunique()} animals")
    print(f"  duration LS {Ld.mean():.1f} +- {Ld.std():.1f} ms, CA1 {Cd.mean():.1f} +- {Cd.std():.1f} ms; rank-sum p = {pd_.pvalue:.2g}; "
          f"medians {Ld.median():.1f} / {Cd.median():.1f}")
    print(f"  frequency LS {L.peak_frequency.mean():.1f} +- {L.peak_frequency.std():.1f} Hz, CA1 {C.peak_frequency.mean():.1f} +- "
          f"{C.peak_frequency.std():.1f} Hz; rank-sum p = {pf.pvalue:.2g}")
print("Animals with both:", animals_both)
L, C = DUR["(1) recordings with both"]
print("\nPer animal (reading 1): mean duration (ms) and frequency (Hz)")
print(pd.DataFrame({"LS dur": L.assign(d=(L.stop_time-L.start_time)*1000).groupby("subject").d.mean(),
                    "CA1 dur": C.assign(d=(C.stop_time-C.start_time)*1000).groupby("subject").d.mean(),
                    "LS freq": L.groupby("subject").peak_frequency.mean(), "CA1 freq": C.groupby("subject").peak_frequency.mean(),
                    "LS n": L.groupby("subject").size(), "CA1 n": C.groupby("subject").size()}).round(1).to_string())
fig, ax = plt.subplots(1, 2, figsize=(9, 3))
ax[0].hist((L.stop_time-L.start_time)*1000, np.arange(0, 120, 2), alpha=.6, density=True, label="LS HFO")
ax[0].hist((C.stop_time-C.start_time)*1000, np.arange(0, 120, 2), alpha=.6, density=True, label="CA1 ripple"); ax[0].set_xlabel("duration (ms)"); ax[0].legend()
ax[1].hist(L.peak_frequency, np.arange(120, 200, 1), alpha=.6, density=True); ax[1].hist(C.peak_frequency, np.arange(120, 200, 1), alpha=.6, density=True)
ax[1].set_xlabel("peak frequency (Hz)"); plt.tight_layout(); plt.show()
print(f"Outside the duration axis (>120 ms): LS {((L.stop_time-L.start_time)*1000 > 120).sum()}, CA1 {((C.stop_time-C.start_time)*1000 > 120).sum()}")

**Result: fig1f-shorter-than-ripples reproduced.** LS HFOs last 27.8 +- 9.5 ms and CA1 ripples 31.2 +- 13.7 ms (paper 27 +- 9 and 31 +- 14), rank-sum p = 1.2e-199, in the four animals with both tables; reading (2) gives the same means. LS HFOs are shorter in each of the four animals.

**Result: fig1g-slower-than-ripples reproduced.** LS HFOs peak at 147.6 +- 5.8 Hz and CA1 ripples at 158.3 +- 7.4 Hz (paper 147 +- 5.7 and 159 +- 8), rank-sum p < 1e-300 (printed as 0); slower in each animal. The frequencies are the lab's deposited values.

## fig1h-no-amplitude-frequency-correlation
> LS HFOs show no significant correlation between event amplitude and peak frequency.

Amplitude is the maximum NSS of the event (claims file; STAR Methods, Ripple detection). The deposited column for it is
`peak_normed_power`, which bz_FindRipples, the lab's detector, fills with the peak of the normalized squared signal
(`peakNormedPower`); the LFP notebook checks it against the maximum NSS computed from the detection channel (claim:fig1h-no-amplitude-frequency-correlation
there). The deposited `peak_amplitude` (the filtered signal's amplitude at the peak) is shown as a variant. All LS HFOs pooled;
Pearson and Spearman at p < 0.05.

In [ ]:
LA = pooled(ls_ev, S)
for col in ["peak_normed_power", "peak_amplitude"]:
    pr = stats.pearsonr(LA[col], LA.peak_frequency); sr = stats.spearmanr(LA[col], LA.peak_frequency)
    print(f"{col} against peak_frequency, {len(LA)} LS HFOs: Pearson r = {pr.statistic:.3f} (p = {pr.pvalue:.2g}); "
          f"Spearman rho = {sr.statistic:.3f} (p = {sr.pvalue:.2g})")
print(f"peak_normed_power: {LA.peak_normed_power.min():.2f} to {LA.peak_normed_power.max():.2f} (median {LA.peak_normed_power.median():.2f}); "
      f"peak_amplitude: {LA.peak_amplitude.min():.1f} to {LA.peak_amplitude.max():.1f}; Spearman between the two {stats.spearmanr(LA.peak_normed_power, LA.peak_amplitude).statistic:.3f}")
for col in ["peak_normed_power", "peak_amplitude"]:
    print(f"\nPer animal ({col}):")
    for sub, g in LA.groupby("subject"):
        pr = stats.pearsonr(g[col], g.peak_frequency); sr = stats.spearmanr(g[col], g.peak_frequency)
        print(f"  {sub}: n = {len(g)}, Pearson r = {pr.statistic:.3f} (p = {pr.pvalue:.2g}), Spearman rho = {sr.statistic:.3f} (p = {sr.pvalue:.2g})")

**Result: fig1h-no-amplitude-frequency-correlation reproduced, resting on an assumption.** Over 27014 LS HFOs, the deposited maximum NSS (`peak_normed_power`) does not correlate with the peak frequency: Pearson r = -0.000 (p = 0.96), Spearman rho = 0.006 (p = 0.36). The variant `peak_amplitude` gives Pearson r = 0.006 (p = 0.3) and a significant but negligible Spearman rho = 0.020 (p = 0.00095). Within animals, `peak_normed_power` correlates significantly by at least one test in 4 of 6 animals (DT2, DT5, DT7, DT8; r from -0.087 to 0.124), with signs that differ between animals. The assumption is that the deposited `peak_normed_power` is the criterion's maximum NSS: the maximum NSS recomputed from the detection channel in the LFP notebook correlates weakly but significantly with the frequency (see there).

## fig1c-hfo-each-animal, fig1c-post-burst-suppression
> Transient 120-180 Hz HFOs occur in LS in each of the 6 animals, with a robust increase in the firing probability of LS neurons at the events.
> LS HFOs are accompanied by a suppression of LS firing immediately after the population burst.

Per LS neuron, the spike probability per event in 1 ms bins over +-100 ms around the LS HFO peak time, and the same at an
equal number of random times in the same recording (uniform over the recording; the claim's seed). Bounds are 3 SEM over neurons.
At the event: the event-time bin and +-5 ms around it. After the burst: (i) from 20 to 50 ms after the event time; (ii) per event,
from its stop to 50 ms after its time.

In [ ]:
def hfo_triggered(seed):
    rng = np.random.default_rng(seed)
    out = []
    for s in S:
        e = ls_ev(s); ls = s.units_in(["ls"])
        if e is None or len(ls) == 0: continue
        t = e.peak_time.to_numpy(); rt = np.sort(rng.uniform(0.2, s.duration - 0.2, len(t)))
        stop_off = e.stop_time.to_numpy() - t
        for u in ls:
            sp = s.spikes[u]
            c = T.triggered_counts(sp, t, 0.1, 0.001).mean(0); cr = T.triggered_counts(sp, rt, 0.1, 0.001).mean(0)
            # (ii) per event: spikes from stop to +50 ms, per ms
            post_ii = (T.count_in(sp, t + stop_off, t + 0.05) / np.maximum((0.05 - stop_off) * 1000, 1)).mean()
            post_ii_r = (T.count_in(sp, rt + stop_off, rt + 0.05) / np.maximum((0.05 - stop_off) * 1000, 1)).mean()
            out.append(dict(subject=s.subject, session=s.session, unit=u, n_ev=len(t), c=c, cr=cr, post_ii=post_ii, post_ii_r=post_ii_r))
    return pd.DataFrame(out)
TRIG = hfo_triggered(seed_of("fig1c-hfo-each-animal"))
lag = np.arange(-100, 101)
at0, at5, post = lag == 0, np.abs(lag) <= 5, (lag >= 20) & (lag <= 50)
def summary(df):
    C, R = np.vstack(df.c), np.vstack(df.cr)
    def ms(x): return x.mean(), x.std(ddof=1) / np.sqrt(len(x))
    r = {}
    for nm, m in [("event bin", at0), ("+-5 ms", at5), ("+20..50 ms", post)]:
        a, b = C[:, m].mean(1), R[:, m].mean(1)
        (ma, sa), (mb, sb) = ms(a), ms(b)
        r[nm] = (ma, sa, mb, sb)
    (ma, sa), (mb, sb) = ms(df.post_ii.to_numpy()), ms(df.post_ii_r.to_numpy())
    r["stop..+50 ms"] = (ma, sa, mb, sb)
    return r
freq_med = LA.groupby("subject").peak_frequency.median()
res1c = []
for sub, g in TRIG.groupby("subject"):
    r = summary(g)
    row = dict(subject=sub, neurons=len(g), events=int(g.drop_duplicates("session").n_ev.sum()), median_freq=round(freq_med.get(sub, np.nan), 1))
    for nm, (ma, sa, mb, sb) in r.items():
        row[f"{nm} HFO"] = ma; row[f"{nm} random"] = mb; row[f"{nm} 3SEM"] = 3 * np.hypot(sa, sb) if False else 3 * sa
        row[f"{nm} above"] = ma - mb > 3 * sa if "+20" not in nm and "stop" not in nm else None
        row[f"{nm} below"] = mb - ma > 3 * sa if ("+20" in nm or "stop" in nm) else None
    res1c.append(row)
R1C = pd.DataFrame(res1c)
pd.set_option("display.float_format", lambda v: f"{v:.5f}")
print(R1C[["subject", "neurons", "events", "median_freq", "event bin HFO", "event bin random", "event bin 3SEM", "event bin above",
           "+-5 ms HFO", "+-5 ms random", "+-5 ms 3SEM", "+-5 ms above"]].to_string())
print("\nAfter the burst (spikes per ms per event):")
print(R1C[["subject", "+20..50 ms HFO", "+20..50 ms random", "+20..50 ms 3SEM", "+20..50 ms below",
           "stop..+50 ms HFO", "stop..+50 ms random", "stop..+50 ms 3SEM", "stop..+50 ms below"]].to_string())
pd.reset_option("display.float_format")
allr = summary(TRIG)
print(f"\nLS neurons in the triggered averages: {len(TRIG)} in {TRIG.session.nunique()} recordings")
print("All animals pooled (mean HFO, SEM, mean random, SEM):", {k: tuple(round(x, 5) for x in v) for k, v in allr.items()})
for nm in ["+20..50 ms", "stop..+50 ms"]:
    ma, sa, mb, sb = allr[nm]
    print(f"Pooled over all LS neurons, after the burst ({nm}): HFO {ma:.6f} +- {sa:.6f} (SEM), random {mb:.6f}; random minus HFO {mb - ma:.6f}, "
          f"3 SEM {3 * sa:.6f}; below by more than 3 SEM: {mb - ma > 3 * sa}")
print(f"Animals with median LS HFO frequency in 120-180 Hz: {int(freq_med.between(120, 180).sum())} of {len(freq_med)}")
print(f"Animals with the event-bin probability above random by more than 3 SEM: {int(R1C['event bin above'].sum())} of {len(R1C)}; "
      f"at +-5 ms: {int(R1C['+-5 ms above'].sum())}")
print(f"Animals with suppression after the burst, +20..50 ms: {int(R1C['+20..50 ms below'].sum())}; stop..+50 ms: {int(R1C['stop..+50 ms below'].sum())}")
# Post-burst relative to the pre-event baseline, as a reference (not the criterion)
C = np.vstack(TRIG.c); base = C[:, (lag >= -100) & (lag <= -50)].mean()
print(f"Pooled mean probability per ms: -100..-50 ms {base:.5f}; event bin {C[:, at0].mean():.5f}; +20..50 ms {C[:, post].mean():.5f}; random {np.vstack(TRIG.cr).mean():.5f}")
fig, ax = plt.subplots(figsize=(6, 3))
for sub, g in TRIG.groupby("subject"):
    ax.plot(lag, np.vstack(g.c).mean(0) / np.vstack(g.cr).mean(), label=sub)
ax.axhline(1, color="k", lw=.5); ax.set_xlabel("lag from LS HFO peak (ms)"); ax.set_ylabel("spike prob. / random"); ax.legend(fontsize=7); plt.show()
yy = [np.vstack(g.c).mean(0) / np.vstack(g.cr).mean() for _, g in TRIG.groupby("subject")]
print(f"Plotted range: {min(y.min() for y in yy):.2f} to {max(y.max() for y in yy):.2f}")

Seed: the random times are drawn under the claim's seed and four more.

In [ ]:
for k in range(5):
    sd = seed_of("fig1c-hfo-each-animal") + 1000 * k
    tr = TRIG if k == 0 else hfo_triggered(sd)
    n_above = n_post = n_post2 = 0
    for sub, g in tr.groupby("subject"):
        r = summary(g)
        n_above += r["event bin"][0] - r["event bin"][2] > 3 * r["event bin"][1]
        n_post += r["+20..50 ms"][2] - r["+20..50 ms"][0] > 3 * r["+20..50 ms"][1]
        n_post2 += r["stop..+50 ms"][2] - r["stop..+50 ms"][0] > 3 * r["stop..+50 ms"][1]
    ra = summary(tr)
    pooled_post = {nm: ra[nm][2] - ra[nm][0] > 3 * ra[nm][1] for nm in ["+20..50 ms", "stop..+50 ms"]}
    print(f"seed {sd}: animals above at the event bin {n_above}; below after the burst (+20..50) {n_post}, (stop..+50) {n_post2}; "
          f"pooled below by more than 3 SEM: {pooled_post}")

**Result: fig1c-hfo-each-animal partially reproduced.** The median LS HFO frequency lies in 120-180 Hz in all 6 animals (142.9 to 149.2 Hz). The HFO-triggered spike probability exceeds the random-time mean by more than 3 SEM at the event bin in 4 of 6 animals (DT1, DT2, DT8, DT9; not DT5 and DT7), and over +-5 ms in 5 of 6 (not DT5); over the five seeds, 4 or 5 animals at the event bin. Pooled, the probability at the event bin is 0.0072 against 0.0016 at random times.

**Result: fig1c-post-burst-suppression not reproduced, resting on an assumption.** Pooled over 2327 LS neurons, the probability from each event's stop to 50 ms after its time is 0.001517 against 0.001593 at random times, a difference of 0.000076, less than 3 SEM (0.000201). From 20 to 50 ms after the event time it is 0.001399, below random by 0.000200, just over 3 SEM (0.000178), under the claim's seed; under 2 of the 4 other seeds it is not. The window from the stop is adopted as the reading of "immediately following the population burst". Per animal, firing is suppressed only in DT5, under both windows.

## figs1-participation
> On average 16% of LS neurons take part in a given HFO, firing an average of 0.23 spikes per event.

Windows (a) start to stop, (b) +-25 ms of the event time; averaging (i) over all events pooled, (ii) over recordings of
per-recording means. Recordings with LS units and LSRipples.

In [ ]:
part = []
for s in S:
    e = ls_ev(s); ls = s.units_in(["ls"])
    if e is None or len(ls) == 0: continue
    t = e.peak_time.to_numpy()
    for w, (a, b) in {"a": (e.start_time.to_numpy(), e.stop_time.to_numpy()), "b": (t - 0.025, t + 0.025)}.items():
        M = np.vstack([T.count_in(s.spikes[u], a, b) for u in ls])        # units x events
        part.append(dict(session=s.session, subject=s.subject, window=w, n_units=len(ls), n_ev=len(t),
                         frac_ev=(M > 0).mean(0), spk_ev=M.mean(0)))
P = pd.DataFrame(part)
for w in "ab":
    g = P[P.window == w]
    fi = np.concatenate(g.frac_ev.values).mean(); si = np.concatenate(g.spk_ev.values).mean()
    fii = np.mean([x.mean() for x in g.frac_ev]); sii = np.mean([x.mean() for x in g.spk_ev])
    print(f"window ({w}): (i) pooled events: fraction {fi:.3f}, spikes per neuron per event {si:.3f}; "
          f"(ii) mean of {len(g)} recordings: fraction {fii:.3f}, spikes {sii:.3f}")
print("Paper: 0.16 (tolerance 0.016) and 0.23 (0.023)")
g = P[P.window == "a"]
print("\nPer animal, window (a), pooled events:")
print(pd.DataFrame({"fraction": g.groupby("subject").frac_ev.apply(lambda x: np.concatenate(x.values).mean()),
                    "spikes": g.groupby("subject").spk_ev.apply(lambda x: np.concatenate(x.values).mean()),
                    "recordings": g.groupby("subject").size()}).round(3).to_string())
print(f"Units per recording: {P.n_units.min()} to {P.n_units.max()}")

**Result: figs1-participation not reproduced.** Over the 73 recordings with LS units and LSRipples, 5.7% of LS neurons fire in an HFO, with 0.079 spikes per neuron per event (window a, pooled events). The largest values over the four combinations are 10.7% and 0.155 spikes (window b, mean over recordings). The paper reports 16% and 0.23. Per animal (window a) the fraction ranges from 3.3% (DT7) to 12.4% (DT1).

## fig5b-nrem-and-rates
> LS HFOs occur primarily during NREM sleep, and their rates are significantly lower than those of hippocampal SPW-Rs.

Per recording with scored sleep, the LS HFO rate (LSRipples) and the CA1 ripple rate (CA1Ripples) in NREM, in WAKE and over
the scored time. The state of an event is the state at its peak time.

In [ ]:
def rates(s, ev):
    if ev is None or s.sleep is None: return None
    t = ev.peak_time.to_numpy(); st = T.state_at(t, s)
    r = {}
    for lab in ["NREM", "WAKE"]:
        dur = T.total(s.state_intervals(lab)); r[lab] = (st == lab).sum() / dur if dur > 60 else np.nan
    scored = T.total(s.sleep[["start_time", "stop_time"]].to_numpy()); r["all"] = (st != "").sum() / scored
    r["n"] = len(t); r["n_nrem"] = int((st == "NREM").sum())
    return r
RL = pd.DataFrame([dict(session=s.session, subject=s.subject, **rates(s, ls_ev(s))) for s in S if rates(s, ls_ev(s)) is not None])
RC = pd.DataFrame([dict(session=s.session, subject=s.subject, **rates(s, ca1_ev(s))) for s in S if rates(s, ca1_ev(s)) is not None])
print(f"LS HFO rate recordings: {len(RL)}; CA1 ripple rate recordings: {len(RC)}")
for nm, R in [("LS HFO", RL), ("CA1 ripple", RC)]:
    print(f"{nm} rate (events/s), median [IQR]: NREM {R.NREM.median():.4f} [{R.NREM.quantile(.25):.4f}, {R.NREM.quantile(.75):.4f}], "
          f"WAKE {R.WAKE.median():.4f} [{R.WAKE.quantile(.25):.4f}, {R.WAKE.quantile(.75):.4f}], all {R['all'].median():.4f}")
frac_nrem = RL.n_nrem.sum() / RL.n.sum()
d = RL.dropna(subset=["NREM", "WAKE"])
w = stats.wilcoxon(d.NREM, d.WAKE)
print(f"\nPrimarily NREM: {frac_nrem:.3f} of {RL.n.sum()} LS HFOs fall in NREM; NREM rate > WAKE rate in {(d.NREM > d.WAKE).sum()} of {len(d)} recordings, "
      f"signed-rank p = {w.pvalue:.2g}")
print("By animal, fraction of LS HFOs in NREM:", (RL.groupby('subject').n_nrem.sum() / RL.groupby('subject').n.sum()).round(3).to_dict())
for col in ["NREM", "all"]:
    a = stats.mannwhitneyu(RL[col].dropna(), RC[col].dropna(), alternative="two-sided")
    m = RL.merge(RC, on="session", suffixes=("_ls", "_ca1")).dropna(subset=[f"{col}_ls", f"{col}_ca1"])
    b = stats.wilcoxon(m[f"{col}_ls"], m[f"{col}_ca1"])
    print(f"Lower than CA1 ({col}): (a) rank-sum {len(RL[col].dropna())} LS against {len(RC[col].dropna())} CA1 rates, p = {a.pvalue:.2g}, "
          f"medians {RL[col].median():.4f} / {RC[col].median():.4f}; (b) signed-rank over {len(m)} recordings with both, p = {b.pvalue:.2g}, "
          f"LS lower in {(m[f'{col}_ls'] < m[f'{col}_ca1']).sum()}")

**Result: fig5b-nrem-and-rates reproduced.** 72.9% of the 27014 LS HFOs fall in NREM, and the NREM rate exceeds the WAKE rate in 66 of 73 recordings (signed-rank p = 2.9e-12); DT7 is the exception, with 29.6% of its HFOs in NREM. LS HFO rates are lower than CA1 ripple rates in NREM (medians 0.024 and 0.320 per s; rank-sum p = 1.5e-13, signed-rank over 42 recordings p = 2.9e-10) and over all scored time (rank-sum p = 1.3e-12). The rates come from 77 LS and 47 CA1 recordings, against the paper's 89 and 53.

## fig5d-lag
> LS HFOs are reliably coupled to CA1 ripples, with a median peak-to-peak lag of 6.4 +- 4.2 ms, CA1 leading LS.

Recordings with both LSRipples and CA1Ripples. Lag = LS HFO peak time minus the nearest CA1 ripple peak time within
+-50 ms or +-25 ms. Event time: the deposited peak (trough of the filtered signal). The maximum-NSS reading needs the
LFP and is in the second notebook. Coupling: pooled CCG of CA1 ripples (trigger) with LS HFOs, 5 ms bins over +-200 ms.

In [ ]:
def nearest_lag(ls_t, ca_t):
    j = np.clip(np.searchsorted(ca_t, ls_t), 1, len(ca_t) - 1)
    a, b = ls_t - ca_t[j - 1], ls_t - ca_t[j]
    return np.where(np.abs(a) < np.abs(b), a, b)
LAGS = []
for s in both_rec:
    l, c = ls_ev(s).peak_time.to_numpy(), np.sort(ca1_ev(s).peak_time.to_numpy())
    LAGS.append(dict(session=s.session, subject=s.subject, lag=nearest_lag(l, c) * 1000, n_ls=len(l), n_ca=len(c)))
LG = pd.DataFrame(LAGS)
for win in [50, 25]:
    per = [np.median(x[np.abs(x) <= win]) for x in LG.lag if (np.abs(x) <= win).sum() >= 1]
    allx = np.concatenate([x[np.abs(x) <= win] for x in LG.lag])
    print(f"within +-{win} ms: {len(allx)} pairs in {len(per)} recordings; (a) median over recordings of medians {np.median(per):.2f} ms "
          f"(SD over recordings {np.std(per, ddof=1):.2f}); (b) pooled median {np.median(allx):.2f} ms; positive (CA1 leading) fraction {np.mean(allx > 0):.3f}")
def ccg_events(trig, targ, maxlag=0.2, bs=0.005):
    L = int(round(maxlag / bs)); edges = (np.arange(-L, L + 2) - 0.5) * bs
    lo = np.searchsorted(targ, trig + edges[0]); hi = np.searchsorted(targ, trig + edges[-1])
    d = np.concatenate([targ[a:b] - t for a, b, t in zip(lo, hi, trig)]) if len(trig) else np.array([])
    return np.histogram(d, edges)[0]
cc = sum(ccg_events(np.sort(ca1_ev(s).peak_time.to_numpy()), np.sort(ls_ev(s).peak_time.to_numpy())) for s in both_rec)
lagc = np.arange(-40, 41) * 5
fl = cc[np.abs(lagc) >= 100]; pk = cc[np.abs(lagc) <= 25].max(); pkl = lagc[np.abs(lagc) <= 25][np.argmax(cc[np.abs(lagc) <= 25])]
print(f"Pooled CCG: peak {pk} at {pkl} ms; flanks (100-200 ms) mean {fl.mean():.1f}, SD {fl.std(ddof=1):.1f}; peak above flank mean by "
      f"{(pk - fl.mean()) / fl.std(ddof=1):.1f} SD")
print("Per animal, median lag within +-50 ms (pooled):", {k: round(float(np.median(np.concatenate([x[np.abs(x) <= 50] for x in g.lag]))), 2) for k, g in LG.groupby("subject")})
fig, ax = plt.subplots(figsize=(5, 3)); ax.bar(lagc, cc, width=5); ax.set_xlabel("LS HFO time - CA1 ripple time (ms)"); ax.set_ylabel("count"); plt.show()
print(f"Plotted counts {cc.min()} to {cc.max()}")

**Result: fig5d-lag reproduced.** With the deposited peak times, the median lag of LS HFOs after the nearest CA1 ripple within +-50 ms is 6.20 ms over recording medians and 6.40 ms over 6572 pooled pairs (paper 6.4 +- 4.2), with CA1 leading in 63.9% of pairs; within +-25 ms, 4.80 ms. The pooled cross-correlogram peaks at 0 ms, 37.1 SD above its 100-200 ms flanks. Per animal the pooled median ranges from -7.2 ms (DT7) to 8.0 ms (DT9). The maximum-NSS reading is in the LFP notebook.

## fig5e-state-coupling
> HPC-LS coupling is stronger in NREM sleep than in waking immobility; the fraction of LS HFOs within +-25 ms of a CA1 ripple
> is 0.3 in NREM and 0.15 in WAKE (P < 6.6e-6).

Recordings with sleep states, CA1Ripples and LSRipples. CCG of CA1 ripples with LS HFOs per recording and state, 5 ms bins over
+-200 ms, normalized by the number of CA1 ripples in the state (the state at the ripple's peak). The fraction of LS HFOs (state at
the HFO's peak) with a CA1 ripple within +-25 ms. WAKE is read as all WAKE and as WAKE outside trials. Recordings need at least one
event of each kind in each state. The inset's test is run (a) over recordings, on their fractions (paired and two-sample), and
(b) over all HFOs pooled, a two-sample t-test of each HFO's 0/1 indicator of a CA1 ripple within +-25 ms (NREM HFOs against WAKE
HFOs), with a two-proportion z-test beside it.

In [ ]:
def coupling(wake_mode):
    rows = []
    for s in both_rec:
        if s.sleep is None: continue
        l, c = np.sort(ls_ev(s).peak_time.to_numpy()), np.sort(ca1_ev(s).peak_time.to_numpy())
        sl, sc = T.state_at(l, s), T.state_at(c, s)
        if wake_mode == "outside trials" and s.has_trials:
            tiv = s.trials[["start_time", "stop_time"]].to_numpy()
            sl = np.where((sl == "WAKE") & T.in_intervals(l, tiv), "", sl); sc = np.where((sc == "WAKE") & T.in_intervals(c, tiv), "", sc)
        r = dict(session=s.session, subject=s.subject)
        ok = True
        for st in ["NREM", "WAKE"]:
            cs, ls_ = c[sc == st], l[sl == st]
            if len(cs) == 0 or len(ls_) == 0: ok = False; break
            r[f"ccg_{st}"] = ccg_events(cs, l) / len(cs)
            lag = nearest_lag(ls_, c) * 1000
            r[f"frac_{st}"] = np.mean(np.abs(lag) <= 25); r[f"nls_{st}"] = len(ls_); r[f"k_{st}"] = int(np.sum(np.abs(lag) <= 25))
        if ok: rows.append(r)
    return pd.DataFrame(rows)
for wm in ["all WAKE", "outside trials"]:
    CP = coupling(wm)
    N, W = np.vstack(CP.ccg_NREM), np.vstack(CP.ccg_WAKE)
    nb = N.shape[1]
    pp = np.minimum(stats.ttest_rel(N, W, axis=0).pvalue * nb, 1); pi = np.minimum(stats.ttest_ind(N, W, axis=0).pvalue * nb, 1)
    near = np.abs(lagc) <= 25
    print(f"WAKE = {wm}: {len(CP)} recordings")
    print(f"  CCG bins within +-25 ms with NREM > WAKE at Bonferroni p < 0.001: paired {int(((pp < 0.001) & (N.mean(0) > W.mean(0)) & near).sum())}, "
          f"two-sample {int(((pi < 0.001) & (N.mean(0) > W.mean(0)) & near).sum())} of {near.sum()}; smallest corrected p paired {pp[near].min():.2g}, two-sample {pi[near].min():.2g}")
    fa, fb = CP.frac_NREM.mean(), CP.frac_WAKE.mean()
    pa, pb = CP.k_NREM.sum() / CP.nls_NREM.sum(), CP.k_WAKE.sum() / CP.nls_WAKE.sum()
    tr, ti = stats.ttest_rel(CP.frac_NREM, CP.frac_WAKE), stats.ttest_ind(CP.frac_NREM, CP.frac_WAKE)
    print(f"  fraction within +-25 ms: (a) mean over recordings NREM {fa:.3f}, WAKE {fb:.3f}; (b) pooled NREM {pa:.3f} ({CP.nls_NREM.sum()} HFOs), "
          f"WAKE {pb:.3f} ({CP.nls_WAKE.sum()} HFOs)")
    print(f"  t-test NREM against WAKE fractions: paired p = {tr.pvalue:.2g} (t = {tr.statistic:.2f}), two-sample p = {ti.pvalue:.2g} (t = {ti.statistic:.2f}); "
          f"NREM higher in {(CP.frac_NREM > CP.frac_WAKE).sum()} of {len(CP)}")
    kN, nN, kW, nW = int(CP.k_NREM.sum()), int(CP.nls_NREM.sum()), int(CP.k_WAKE.sum()), int(CP.nls_WAKE.sum())
    tp = stats.ttest_ind(np.r_[np.ones(kN), np.zeros(nN - kN)], np.r_[np.ones(kW), np.zeros(nW - kW)])
    pp_ = (kN + kW) / (nN + nW); zz = (pa - pb) / np.sqrt(pp_ * (1 - pp_) * (1 / nN + 1 / nW))
    print(f"  (b) pooled over HFOs: two-sample t-test p = {tp.pvalue:.2g} (t = {tp.statistic:.1f}, {nN} NREM and {nW} WAKE HFOs); "
          f"two-proportion z = {zz:.1f}, p = {2 * stats.norm.sf(abs(zz)):.2g}")
    print("  per animal, pooled fractions NREM / WAKE (HFOs):", {k: (round(g.k_NREM.sum() / g.nls_NREM.sum(), 3), round(g.k_WAKE.sum() / max(g.nls_WAKE.sum(), 1), 3),
          int(g.nls_NREM.sum()), int(g.nls_WAKE.sum())) for k, g in CP.groupby("subject")})
    if wm == "all WAKE": CP_ALL = CP
fig, ax = plt.subplots(figsize=(5, 3))
ax.plot(lagc, np.vstack(CP_ALL.ccg_NREM).mean(0), label="NREM"); ax.plot(lagc, np.vstack(CP_ALL.ccg_WAKE).mean(0), label="WAKE")
ax.set_xlabel("lag (ms)"); ax.set_ylabel("LS HFOs per CA1 ripple per bin"); ax.legend(); plt.show()

**Result: fig5e-state-coupling partially reproduced, resting on an assumption.** Over 42 recordings, the fraction of LS HFOs with a CA1 ripple within +-25 ms is 0.302 in NREM and 0.091 in WAKE with all HFOs pooled (paper 0.3 and 0.15), and 0.213 and 0.118 as the mean over recordings. Pooled over HFOs (15605 NREM and 4163 WAKE HFOs) the NREM fraction is higher at t-test p = 3.1e-171, below 6.6e-6; over recordings it is higher in 30 of 42, with p = 0.0025 paired and 0.014 two-sample, short of 0.001. No cross-correlogram bin within +-25 ms differs at corrected p < 0.001 (smallest 0.026), and the WAKE fraction is outside tolerance under both readings, so the outcome is partial under the pooled reading and not reproduced under the per-recording one. The pooled reading is adopted as an assumption: the paper's NREM fraction of 0.3 matches the pooled 0.302, not the mean over recordings (0.213). Excluding trials from WAKE changes nothing (pooled WAKE 0.092, p = 3.7e-169).

## fig5f-ripple-triggered-firing
> LS firing around CA1 ripples is higher during NREM than during waking.

Per LS neuron in recordings with sleep states and CA1Ripples, the CA1 ripple-triggered rate (spikes/s) in 5 ms bins over +-200 ms,
for NREM and WAKE ripples separately; neurons with at least one ripple in each state. Per-bin t-tests over neurons, Bonferroni
over 81 bins; paired and two-sample; raw rates and rates divided by the neuron's mean rate in the state. WAKE is read as all WAKE and, as in fig5e-state-coupling,
as WAKE outside trials (ripples and the mean rate both outside trials).

In [ ]:
F5 = []
for s in S:
    c = ca1_ev(s); ls = s.units_in(["ls"])
    if c is None or s.sleep is None or len(ls) == 0: continue
    t = c.peak_time.to_numpy(); st = T.state_at(t, s)
    if (st == "NREM").sum() == 0 or (st == "WAKE").sum() == 0: continue
    tiv = s.trials[["start_time", "stop_time"]].to_numpy() if s.has_trials else np.zeros((0, 2))
    wo = (st == "WAKE") & ~T.in_intervals(t, tiv)
    wk = s.state_intervals("WAKE"); mt = T.merge_intervals(tiv) if len(tiv) else tiv
    gaps = np.column_stack([np.r_[0.0, mt[:, 1]], np.r_[mt[:, 0], s.duration]]) if len(mt) else np.array([[0.0, s.duration]])
    wk_out = T.intersect(wk, gaps[gaps[:, 1] > gaps[:, 0]])          # WAKE outside the trial intervals
    for u in ls:
        r = dict(session=s.session, subject=s.subject)
        for lab in ["NREM", "WAKE"]:
            r[lab] = T.triggered_counts(s.spikes[u], t[st == lab], 0.2, 0.005).mean(0) / 0.005
            iv = s.state_intervals(lab); r[f"mean_{lab}"] = T.in_intervals(s.spikes[u], iv).sum() / max(T.total(iv), 1)
        r["WAKEo"] = T.triggered_counts(s.spikes[u], t[wo], 0.2, 0.005).mean(0) / 0.005 if wo.sum() else np.full(81, np.nan)
        r["mean_WAKEo"] = T.in_intervals(s.spikes[u], wk_out).sum() / max(T.total(wk_out), 1)
        F5.append(r)
F5 = pd.DataFrame(F5)
N, W = np.vstack(F5.NREM), np.vstack(F5.WAKE)
mN, mW = F5.mean_NREM.to_numpy()[:, None], F5.mean_WAKE.to_numpy()[:, None]
okn = (mN[:, 0] > 0) & (mW[:, 0] > 0)
print(f"LS neurons: {len(F5)} in {F5.session.nunique()} recordings, {F5.subject.nunique()} animals; with a nonzero rate in both states {okn.sum()}")
near = np.abs(lagc) <= 25
for nm, (a, b) in {"raw": (N, W), "normalized": (N[okn] / mN[okn], W[okn] / mW[okn])}.items():
    pp = np.minimum(stats.ttest_rel(a, b, axis=0).pvalue * 81, 1); pi = np.minimum(stats.ttest_ind(a, b, axis=0).pvalue * 81, 1)
    hi = a.mean(0) > b.mean(0)
    print(f"{nm}: bins within +-25 ms with NREM higher at corrected p < 0.001: paired {int(((pp < .001) & hi & near).sum())}, two-sample "
          f"{int(((pi < .001) & hi & near).sum())} of {near.sum()}; at 0 lag NREM {a[:, 40].mean():.2f}, WAKE {b[:, 40].mean():.2f}; "
          f"smallest corrected p within +-25 ms paired {pp[near].min():.2g}, two-sample {pi[near].min():.2g}; bins anywhere with WAKE higher (paired p<0.001): {int(((pp < .001) & ~hi).sum())}")
Wo, mWo = np.vstack(F5.WAKEo), F5.mean_WAKEo.to_numpy()[:, None]
oko = ~np.isnan(Wo).any(1) & (mN[:, 0] > 0) & (mWo[:, 0] > 0)
print(f"WAKE outside trials: {oko.sum()} neurons with a ripple in WAKE outside trials and nonzero rates")
for nm, (a, b) in {"raw": (N[oko], Wo[oko]), "normalized": (N[oko] / mN[oko], Wo[oko] / mWo[oko])}.items():
    pp = np.minimum(stats.ttest_rel(a, b, axis=0).pvalue * 81, 1); pi = np.minimum(stats.ttest_ind(a, b, axis=0).pvalue * 81, 1)
    hi = a.mean(0) > b.mean(0)
    print(f"  {nm}, WAKE outside trials: bins within +-25 ms with NREM higher at corrected p < 0.001: paired {int(((pp < .001) & hi & near).sum())}, two-sample "
          f"{int(((pi < .001) & hi & near).sum())} of {near.sum()}; at 0 lag NREM {a[:, 40].mean():.2f}, WAKE {b[:, 40].mean():.2f}; "
          f"smallest corrected p within +-25 ms with NREM higher, paired {np.min(np.where(hi, pp, 1)[near]):.2g}, two-sample {np.min(np.where(hi, pi, 1)[near]):.2g}")
print("Per animal, mean rate at 0 lag (raw) NREM / WAKE:", {k: (round(np.vstack(g.NREM)[:, 40].mean(), 2), round(np.vstack(g.WAKE)[:, 40].mean(), 2), len(g)) for k, g in F5.groupby("subject")})
fig, ax = plt.subplots(figsize=(5, 3)); ax.plot(lagc, N.mean(0), label="NREM"); ax.plot(lagc, W.mean(0), label="WAKE"); ax.set_xlabel("lag from CA1 ripple (ms)"); ax.set_ylabel("LS rate (Hz)"); ax.legend(); plt.show()
print(f"Plotted range {min(N.mean(0).min(), W.mean(0).min()):.2f} to {max(N.mean(0).max(), W.mean(0).max()):.2f} Hz")

**Result: fig5f-ripple-triggered-firing not reproduced.** Over 1266 LS neurons in 38 recordings (paper 1313), the CA1 ripple-triggered rate at zero lag is 2.14 Hz in NREM and 2.03 Hz in WAKE; no bin within +-25 ms has NREM higher at corrected p < 0.001 under any reading (smallest corrected p 0.18 paired, normalized rates). Normalized by the state's mean rate the WAKE response is the larger (1.73 against 1.48). With WAKE outside trials (1264 neurons) the same holds: 2.14 against 2.04 Hz raw, 1.48 against 1.75 normalized, no bin with NREM higher at corrected p < 0.001 (smallest 0.59).

## discussion-medial-septum (part b) and discussion-cingulate-down-state (reading a)
> Recordings in medial septum show no HFOs, and medial septal firing is mainly suppressed at hippocampal SPW-Rs.
> Recordings in deep layers of anterior cingulate cortex show down-state-coupled activity about 50 ms after the CA1 SPW-R.

Units labelled `ms` and `acg`. Part (b): CA1 ripple-triggered rate (all states) within +-50 ms of the ripple against the
neuron's mean rate; signed-rank test. Cingulate (a): the time of the minimum of the mean triggered rate (5 ms bins) between 0 and 200 ms.
Part (a) of the medial septum claim needs the LFP of many sites and is in the second notebook.

In [ ]:
for reg in ["ms", "acg"]:
    rows = []
    for s in S:
        c = ca1_ev(s); us = s.units_in([reg])
        if c is None or len(us) == 0: continue
        t = c.peak_time.to_numpy()
        for u in us:
            sp = s.spikes[u]; mr = len(sp) / s.duration
            tc = T.triggered_counts(sp, t, 0.2, 0.005).mean(0) / 0.005
            r50 = (T.count_in(sp, t - 0.05, t + 0.05) / 0.1).mean()
            rows.append(dict(session=s.session[:25], subject=s.subject, mean_rate=mr, rate50=r50, tc=tc))
    D = pd.DataFrame(rows)
    print(f"{reg}: {len(D)} units in {D.session.nunique()} recordings ({D.session.unique().tolist()})")
    if reg == "ms":
        w = stats.wilcoxon(D.rate50, D.mean_rate)
        print(f"  rate within +-50 ms of CA1 ripples below the mean rate in {(D.rate50 < D.mean_rate).sum()} of {len(D)}; median ratio "
              f"{np.median(D.rate50 / D.mean_rate):.2f}; signed-rank p = {w.pvalue:.2g}")
    else:
        m = np.vstack(D.tc).mean(0) / D.mean_rate.mean()
        win = (lagc >= 0) & (lagc <= 200)
        print(f"  pooled triggered rate / mean rate: minimum in 0-200 ms at {lagc[win][np.argmin(m[win])]} ms ({m[win].min():.2f}); maximum at "
              f"{lagc[win][np.argmax(m[win])]} ms ({m[win].max():.2f}); at 0 ms {m[40]:.2f}")
        print("  per recording minimum lag (ms):", {k: int(lagc[win][np.argmin(np.vstack(g.tc).mean(0)[win])]) for k, g in D.groupby("session")})
        fig, ax = plt.subplots(figsize=(5, 3)); ax.plot(lagc, m); ax.set_xlabel("lag from CA1 ripple (ms)"); ax.set_ylabel("ACC rate / mean"); plt.show()
        print(f"Plotted range {m.min():.2f} to {m.max():.2f}")

**Result: discussion-medial-septum, part (b), does not hold.** The 25 units labelled `ms` (2 recordings, DT7) fire more within +-50 ms of CA1 ripples than on average (median ratio 1.43; below the mean in 6 of 25; signed-rank p = 0.0046). Part (a) and the outcome are in the LFP notebook.

**Result: discussion-cingulate-down-state, reading (a), does not hold.** The 31 units labelled `acg` (2 recordings, DT7) reach their minimum rate after CA1 ripples at 105 ms (0.62 of the mean rate), against 50 +- 5 ms, and a maximum at 150 ms; per recording the minimum lies at 105 and 130 ms. Reading (b) is in the LFP notebook.

## Figure 4: LS pairs during HFOs and on the maze
Common to fig4b-pre-post-unchanged, fig4b-hfo-above-behav, fig4-sessions-few-change, fig4c-temporal-bias and fig4d-compression.
Recordings with trials (for the epochs; see the helper's docstring for the PRE / BEHAV / POST assumption), LSRipples and LS
units on at least two shanks. Pairs of LS neurons on different shanks. Spikes during LS HFOs (start to stop) in PRE and in
POST; BEHAV spikes inside trials (reading iv-a) or anywhere from the first trial to the last (iv-b). CCGs in 1 ms bins over
+-200 ms. Inclusion (i): (a) the CCG holds more than 5 spike pairs within +-200 ms, (b) each neuron has more than 5 spikes
with a partner spike within 200 ms; in every epoch.

In [ ]:
def partner_counts(a, b, w=0.2):
    """Spikes of a with a spike of b within w, and of b with one of a."""
    if len(a) == 0 or len(b) == 0: return 0, 0
    def k(x, y):
        j = np.clip(np.searchsorted(y, x), 1, len(y) - 1) if len(y) > 1 else np.zeros(len(x), int)
        d = np.minimum(np.abs(y[j] - x), np.abs(y[np.maximum(j - 1, 0)] - x))
        return int((d <= w).sum())
    return k(a, b), k(b, a)
def epoch_spikes(s, units, kind, ev_fn, behav_mode, spikes=None):
    sp = [s.spikes[u] for u in units] if spikes is None else spikes
    ep = s.epochs("trials" if behav_mode == "trials" else "span")
    e = ev_fn(s); hfo = e[["start_time", "stop_time"]].to_numpy()
    return {"PRE": T.restrict(sp, T.intersect(hfo, ep["PRE"])), "BEHAV": T.restrict(sp, ep["BEHAV"]),
            "POST": T.restrict(sp, T.intersect(hfo, ep["POST"]))}
def session_pairs(s, units, ev_fn, behav_mode, cross_shank=True, spikes=None):
    """CCGs (+-200 ms, 1 ms) of every pair in the three epochs, and both inclusion flags."""
    es = epoch_spikes(s, units, None, ev_fn, behav_mode, spikes)
    C = {k: T.ccg_all(v, 0.2) for k, v in es.items()}
    n = len(units); sh = s.unit_shank[units]
    I, J = np.triu_indices(n, 1)
    keep = sh[I] != sh[J] if cross_shank else np.ones(len(I), bool)
    I, J = I[keep], J[keep]
    inc_a_ep = {k: C[k][I, J].sum(1) > 5 for k in C}
    inc_a = np.all([inc_a_ep[k] for k in C], axis=0)
    inc_b = np.ones(len(I), bool); inc_b_ep = {}
    for k in C:
        pc = np.array([partner_counts(es[k][i], es[k][j]) for i, j in zip(I, J)]).reshape(-1, 2)
        inc_b_ep[k] = (pc[:, 0] > 5) & (pc[:, 1] > 5); inc_b &= inc_b_ep[k]
    nsp = {k: np.array([len(x) for x in v]) for k, v in es.items()}
    return dict(I=I, J=J, ccg={k: C[k][I, J].astype(float) for k in C}, inc_a=inc_a, inc_b=inc_b, inc_a_ep=inc_a_ep, inc_b_ep=inc_b_ep,
                nspk={k: int(v.sum()) for k, v in nsp.items()},
                norm={k: np.sqrt(np.maximum(nsp[k][I] * nsp[k][J], 1)) for k in C})
F4_SESS = [s for s in S if s.has_trials and ls_ev(s) is not None and len(set(s.unit_shank[s.units_in(["ls"])])) >= 2]
print(f"Recordings with trials, LSRipples and LS units on two or more shanks: {len(F4_SESS)} ({collections.Counter(s.subject for s in F4_SESS)})")
t0 = time.time()
PAIRS = {}
for bm in ["trials", "span"]:
    PAIRS[bm] = [(s, session_pairs(s, s.units_in(["ls"]), ls_ev, bm)) for s in F4_SESS]
print(f"CCGs computed in {time.time()-t0:.0f} s")
def excess(ccg, kind):
    return T.gauss_smooth(ccg, 5, kind)[:, 200] - T.gauss_smooth(ccg, 200, kind)[:, 200]
EXC = []
for bm, lst in PAIRS.items():
    for s, P in lst:
        for inc in ["a", "b"]:
            m = P[f"inc_{inc}"]
            for kind in ["sd", "fwhm"]:
                for nm in ["raw", "sqrt"]:
                    r = dict(behav=bm, inc=inc, kernel=kind, norm=nm, session=s.session, subject=s.subject, n=int(m.sum()))
                    for ep in ["PRE", "BEHAV", "POST"]:
                        c = P["ccg"][ep][m]
                        if nm == "sqrt": c = c / P["norm"][ep][m][:, None]
                        r[ep] = excess(c, kind) if len(c) else np.array([])
                    EXC.append(r)
EXC = pd.DataFrame(EXC)
rows = []
for key, g in EXC.groupby(["inc", "kernel", "norm", "behav"]):
    pre, post, beh = (np.concatenate(g[e].values) for e in ["PRE", "POST", "BEHAV"])
    p = stats.mannwhitneyu(pre, post).pvalue if len(pre) else np.nan
    rows.append(dict(inclusion=key[0], kernel=key[1], norm=key[2], behav=key[3], pairs=len(pre), sessions=int((g.n > 0).sum()),
                     PRE_median=np.median(pre), POST_median=np.median(post), BEHAV_median=np.median(beh),
                     PRE_mean=np.mean(pre), POST_mean=np.mean(post), BEHAV_mean=np.mean(beh), p_PRE_POST=p,
                     differ=p < 0.05, effect_r=(stats.mannwhitneyu(pre, post).statistic / (len(pre) * len(post)) - 0.5) * 2 if len(pre) else np.nan))
F4B = pd.DataFrame(rows)
print(F4B.to_string(float_format=lambda v: f"{v:.4g}"))
print(f"\nReadings where PRE and POST differ (rank-sum p < 0.05): {int(F4B.differ.sum())} of {len(F4B)}; pairs per reading {F4B.pairs.min()} to {F4B.pairs.max()} "
      f"(paper 7172, tolerance 717); sessions {F4B.sessions.min()} to {F4B.sessions.max()} (paper 53)")
g = EXC[(EXC.inc == "a") & (EXC.kernel == "sd") & (EXC.norm == "raw") & (EXC.behav == "trials")]
print("\nPer animal, reading (a, sd, raw, trials): pairs, median PRE, median POST, rank-sum p")
for sub, gg in g.groupby("subject"):
    pre, post = np.concatenate(gg.PRE.values), np.concatenate(gg.POST.values)
    print(f"  {sub}: {len(pre)}, {np.median(pre) if len(pre) else np.nan:.3g}, {np.median(post) if len(post) else np.nan:.3g}, "
          f"{stats.mannwhitneyu(pre, post).pvalue if len(pre) > 1 else np.nan:.2g}")

In [ ]:
# Where the pairs are lost (verifier round 1): per recording, the candidate cross-shank LS pairs, the pairs passing inclusion (a)
# in each epoch and in all three, the LS HFOs and their total duration in PRE and POST, and the LS spikes inside them (BEHAV in trials).
rows = []
for s, P in PAIRS["trials"]:
    ep = s.epochs("trials"); e = ls_ev(s); hfo = e[["start_time", "stop_time"]].to_numpy()
    r = dict(subject=s.subject, session=s.session[:24], ls_units=len(s.units_in(["ls"])), shanks=len(set(s.unit_shank[s.units_in(["ls"])])), pairs=len(P["I"]))
    for k in ["PRE", "BEHAV", "POST"]: r[f"a_{k}"] = int(P["inc_a_ep"][k].sum())
    r["a_all"] = int(P["inc_a"].sum()); r["b_all"] = int(P["inc_b"].sum())
    for k in ["PRE", "POST"]:
        h = T.intersect(hfo, ep[k]); r[f"hfo_{k}"] = len(h); r[f"hfo_s_{k}"] = round(T.total(h), 1); r[f"{k}_min"] = round(T.total(ep[k]) / 60, 1)
        r[f"spk_{k}"] = P["nspk"][k]
    rows.append(r)
F4D = pd.DataFrame(rows)
print(F4D.to_string())
print(f"\nTotals: candidate pairs {F4D.pairs.sum()}; passing (a) in PRE {F4D.a_PRE.sum()}, BEHAV {F4D.a_BEHAV.sum()}, POST {F4D.a_POST.sum()}, all three {F4D.a_all.sum()} "
      f"(inclusion b {F4D.b_all.sum()}); recordings with a passing pair {int((F4D.a_all > 0).sum())} of {len(F4D)}")
print(f"LS HFOs in PRE {F4D.hfo_PRE.sum()} ({F4D.hfo_s_PRE.sum():.0f} s) and in POST {F4D.hfo_POST.sum()} ({F4D.hfo_s_POST.sum():.0f} s); median per recording "
      f"{F4D.hfo_PRE.median():.0f} and {F4D.hfo_POST.median():.0f}; PRE and POST last a median {F4D.PRE_min.median():.0f} and {F4D.POST_min.median():.0f} min")
_ls = [s for s in S if len(s.units_in(["ls"]))]
print("Recordings with LS units left out of Figure 4, by animal: no trials", dict(collections.Counter(s.subject for s in _ls if not s.has_trials)),
      "; no LSRipples", dict(collections.Counter(s.subject for s in _ls if s.has_trials and ls_ev(s) is None)),
      "; LS units on one shank", dict(collections.Counter(s.subject for s in _ls if s.has_trials and ls_ev(s) is not None and len(set(s.unit_shank[s.units_in(["ls"])])) < 2)))
print("Per animal:", F4D.groupby("subject")[["pairs", "a_PRE", "a_BEHAV", "a_POST", "a_all", "hfo_PRE", "hfo_POST"]].sum().to_dict("index"))

In [ ]:
# Variant not named in the claims file (verifier round 2; asked of a reader in questions/tingley-2019.yaml): the inclusion
# rule applied to all of each epoch's spikes, PRE and POST included, while the CCGs are still those of the spikes during LS HFOs.
# STAR Methods, Excess correlation analysis: CCGs "for each epoch of the recording sessions (PRE, BEHAV, and POST)"; "Only
# spike pairs where both neurons fired more than 5 spikes within 200 milliseconds of each other were included."
def pair_counts_200(a, b, w=0.2):
    """Spike pairs of a and b within +-w (inclusion a over a whole epoch)."""
    if len(a) == 0 or len(b) == 0: return 0
    return int((np.searchsorted(b, a + w, side="right") - np.searchsorted(b, a - w, side="left")).sum())
t0 = time.time()
WHOLE = {}
for s, P in PAIRS["trials"]:
    units = s.units_in(["ls"]); ep = s.epochs("trials"); wa = wb = np.ones(len(P["I"]), bool)
    for k in ["PRE", "POST"]:
        es = T.restrict([s.spikes[u] for u in units], ep[k])
        wa = wa & np.array([pair_counts_200(es[i], es[j]) > 5 for i, j in zip(P["I"], P["J"])], bool)
        pc = np.array([partner_counts(es[i], es[j]) for i, j in zip(P["I"], P["J"])]).reshape(-1, 2)
        wb = wb & (pc[:, 0] > 5) & (pc[:, 1] > 5)
    WHOLE[s.session] = wa, wb
for bm, lst in PAIRS.items():
    for s, P in lst:
        P["inc_wa"] = WHOLE[s.session][0] & P["inc_a_ep"]["BEHAV"]; P["inc_wb"] = WHOLE[s.session][1] & P["inc_b_ep"]["BEHAV"]
print(f"Whole-epoch inclusion computed in {time.time()-t0:.0f} s; pairs passing in PRE and POST: (a) {sum(v[0].sum() for v in WHOLE.values())}, "
      f"(b) {sum(v[1].sum() for v in WHOLE.values())}, of {sum(len(v[0]) for v in WHOLE.values())} candidates")
rows = []
for bm, lst in PAIRS.items():
    for inc in ["wa", "wb"]:
        for kind in ["sd", "fwhm"]:
            for nm in ["raw", "sqrt"]:
                ex = {ep: [] for ep in ["PRE", "BEHAV", "POST"]}; nses = 0; sess_sig = [0, 0, 0, 0]
                for s, P in lst:
                    m = P[f"inc_{inc}"]; nses += m.sum() > 0
                    e = {}
                    for ep in ex:
                        c = P["ccg"][ep][m]
                        if nm == "sqrt": c = c / P["norm"][ep][m][:, None]
                        e[ep] = excess(c, kind) if len(c) else np.array([]); ex[ep].append(e[ep])
                    if len(e["PRE"]) > 1 and stats.mannwhitneyu(e["PRE"], e["POST"]).pvalue < 0.01:
                        sess_sig[0] += 1; sess_sig[1] += np.median(e["PRE"]) < np.median(e["POST"]); sess_sig[2] += np.median(e["PRE"]) > np.median(e["POST"]); sess_sig[3] += np.median(e["PRE"]) == np.median(e["POST"])
                pre, beh, post = (np.concatenate(ex[k]) for k in ["PRE", "BEHAV", "POST"])
                mw = stats.mannwhitneyu(pre, post)
                rows.append(dict(inclusion=inc[1] + " whole", kernel=kind, norm=nm, behav=bm, pairs=len(pre), sessions=int(nses),
                                 p_PRE_POST=mw.pvalue, effect_r=(mw.statistic / (len(pre) * len(post)) - 0.5) * 2,
                                 p_PRE_gt_BEHAV=stats.mannwhitneyu(pre, beh, alternative="greater").pvalue,
                                 p_POST_gt_BEHAV=stats.mannwhitneyu(post, beh, alternative="greater").pvalue,
                                 sessions_differ=sess_sig[0], PRE_below=sess_sig[1], PRE_above=sess_sig[2], medians_equal=sess_sig[3],
                                 zero_PRE=float(np.mean(pre == 0)), zero_POST=float(np.mean(post == 0))))
F4W = pd.DataFrame(rows)
print(F4W.to_string(float_format=lambda v: f"{v:.3g}"))
print(f"\nPairs per reading {F4W.pairs.min()} to {F4W.pairs.max()} (paper 7172 +- 717.2), sessions {F4W.sessions.min()} to {F4W.sessions.max()} (paper 53); "
      f"PRE and POST differ (rank-sum p < 0.05) in {int((F4W.p_PRE_POST < 0.05).sum())} of {len(F4W)} readings "
      f"({int((F4W[F4W.norm == 'sqrt'].p_PRE_POST < 0.05).sum())} of {int((F4W.norm == 'sqrt').sum())} normalized)")

**Result: fig4b-pre-post-unchanged partially reproduced, resting on an assumption.** With the CCG divided by the square root of the two spike counts, no reading finds PRE and POST different (rank-sum p 0.50 to 0.89 over 8 readings; effect r -0.013 to 0.025). With raw counts, all 8 readings find POST above PRE (p 0.0036 to 0.0065; r -0.10 to -0.12). The normalized CCG is adopted as an assumption: the paper compares epochs with very different spike counts, which only a rate-normalized measure allows (in raw counts the mean BEHAV excess is 13.1 to 83.0 against 0.60 to 0.85 in PRE, while the medians are comparable, BEHAV 0.088 to 0.66 and PRE 0.32 to 0.44); POST holds more HFO time than PRE (355 s against 252 s over the 54 recordings), which raw counts carry. Only 361 to 503 pairs in 21 to 23 sessions pass the inclusion rule, against 7172 pairs in 53 sessions. The pairs are lost in the HFO epochs: of 20182 candidate cross-shank pairs, 10823 pass inclusion (a) in BEHAV but only 795 in PRE and 954 in POST, where the LS HFOs last 252 s and 355 s in all; 462 pass in all three. DT8 has 1886 candidate pairs, of which 3 pass in PRE and 10 in POST, and none in all three. Of the recordings with LS units, 17 have no trials, 7 no LSRipples and 4 LS units on one shank. A variant that the claims file does not name, asked of a reader (questions/tingley-2019.yaml), applies the inclusion rule to all spikes of PRE and POST, with the CCGs still from the spikes during HFOs: 19274 of the 20182 candidate pairs pass inclusion (a) in PRE and POST, and with BEHAV 10159 to 15732 pairs in 54 sessions per reading (paper 7172 +- 717.2 in 53). Of these pairs, 73.8% to 76.6% have an excess correlation of exactly 0 in PRE and 70.5% to 73.2% in POST. Under this variant PRE and POST differ (rank-sum p < 0.05) in 8 of 16 readings: in 4 of the 8 normalized readings, all with the 5 ms SD kernel (p 0.00047 to 0.0082), and in none with the full-width kernel (p 0.15 to 0.29); the effects are small (r -0.016 to -0.023 where they differ, POST higher). The outcome is judged on the claims file's readings, so it stays as above; under this variant it would not hold for every normalized reading.

## fig4b-hfo-above-behav
> Excess correlations during HFOs (PRE and POST) are much greater than chance from spikes jittered by +-200 ms and BEHAV excess
> correlations slightly greater (P < 0.01), and co-activation during HFOs is higher than during maze traversal (P < 0.001).

Every spike of each neuron is moved by a uniform random amount in +-200 ms (one jitter), and the same epochs, pairs and measure are
computed. Reported for each reading of fig4b-pre-post-unchanged; the jitter is drawn under the claim's seed and four more.

In [ ]:
def jittered_excess(seed):
    rng = np.random.default_rng(seed)
    out = []
    for bm in ["trials", "span"]:
        for (s, P) in PAIRS[bm]:
            units = s.units_in(["ls"])
            js = T.jitter([s.spikes[u] for u in units], rng)
            es = epoch_spikes(s, units, None, ls_ev, bm, js)
            C = {k: T.ccg_all(v, 0.2)[P["I"], P["J"]].astype(float) for k, v in es.items()}
            nsp = {k: np.array([len(x) for x in v]) for k, v in es.items()}
            for inc in ["a", "b"]:
                m = P[f"inc_{inc}"]
                for kind in ["sd", "fwhm"]:
                    for nm in ["raw", "sqrt"]:
                        r = dict(behav=bm, inc=inc, kernel=kind, norm=nm, session=s.session)
                        for ep in C:
                            c = C[ep][m]
                            if nm == "sqrt": c = c / np.sqrt(np.maximum(nsp[ep][P["I"]] * nsp[ep][P["J"]], 1))[m][:, None]
                            r[ep] = excess(c, kind) if len(c) else np.array([])
                        out.append(r)
    return pd.DataFrame(out)
def above_table(JX):
    rows = []
    for key, g in EXC.groupby(["inc", "kernel", "norm", "behav"]):
        gj = JX[(JX.inc == key[0]) & (JX.kernel == key[1]) & (JX.norm == key[2]) & (JX.behav == key[3])]
        r = dict(inclusion=key[0], kernel=key[1], norm=key[2], behav=key[3])
        x = {e: np.concatenate(g[e].values) for e in ["PRE", "BEHAV", "POST"]}
        xj = {e: np.concatenate(gj[e].values) for e in ["PRE", "BEHAV", "POST"]}
        for a, b in [("PRE", "BEHAV"), ("POST", "BEHAV")]:
            t = stats.mannwhitneyu(x[a], x[b]); r[f"p {a}>{b}"] = t.pvalue; r[f"{a}>{b}"] = (t.pvalue < 0.001) and np.median(x[a]) > np.median(x[b])
        for e in ["PRE", "BEHAV", "POST"]:
            t = stats.mannwhitneyu(x[e], xj[e]); r[f"p {e}>jit"] = t.pvalue
            r[f"{e}>jit"] = (t.pvalue < 0.01) and np.mean(x[e]) > np.mean(xj[e])
            r[f"{e} mean"] = np.mean(x[e]); r[f"{e} jit mean"] = np.mean(xj[e])
        rows.append(r)
    D = pd.DataFrame(rows)
    D["part1"] = D["PRE>BEHAV"] & D["POST>BEHAV"]; D["part2"] = D["PRE>jit"] & D["BEHAV>jit"] & D["POST>jit"]
    return D
JX0 = jittered_excess(seed_of("fig4b-hfo-above-behav"))
F4J = above_table(JX0)
print(F4J.to_string(float_format=lambda v: f"{v:.3g}"))
print(f"\nReadings with both parts: {int((F4J.part1 & F4J.part2).sum())}; part 1 only: {int((F4J.part1 & ~F4J.part2).sum())}; "
      f"part 2 only: {int((~F4J.part1 & F4J.part2).sum())}; neither: {int((~F4J.part1 & ~F4J.part2).sum())} of {len(F4J)}")

In [ ]:
# The same under four more seeds (jitter only).
for k in range(1, 5):
    sd = seed_of("fig4b-hfo-above-behav") + 1000 * k
    D = above_table(jittered_excess(sd))
    print(f"seed {sd}: readings with part 2 (each epoch above its jitter at p < 0.01) {int(D.part2.sum())} of {len(D)}; "
          f"BEHAV above jitter in {int(D['BEHAV>jit'].sum())}; PRE in {int(D['PRE>jit'].sum())}; POST in {int(D['POST>jit'].sum())}")

**Result: fig4b-hfo-above-behav reproduced, resting on an assumption.** Under all 8 normalized readings, PRE and POST exceed BEHAV (rank-sum p 7e-69 to 3e-129) and PRE, POST and BEHAV each exceed their jittered values (p <= 2.1e-6). Under raw counts, every epoch exceeds its jitter; with BEHAV inside trials (reading iv-a) PRE and POST also exceed BEHAV (p <= 4.9e-13), so both parts hold, but with BEHAV taken from the first to the last trial they do not (p 0.13 to 0.87). The outcome therefore fails only under raw counts with BEHAV over the whole span, and rests on the normalized CCG or on BEHAV inside trials. Under the four other jitter seeds, every epoch exceeds its jitter under all 16 readings. Under the whole-epoch inclusion variant of fig4b-pre-post-unchanged (not in the claims file, asked of a reader), PRE and POST both exceed BEHAV at p < 0.001 (one-sided rank-sum) in 10 of 16 readings; not in the 4 raw readings with the 5 ms SD kernel (p 0.95 to 1), and for PRE not in the 2 readings with inclusion (b), the full-width kernel and BEHAV over the whole span (p 0.0015 and 0.0021). The jitter part was not run under this variant.

## fig4-sessions-few-change
> Only 7 of 53 sessions have significantly different excess correlations in PRE and POST (rank-sum P < 0.01), three with PRE below
> POST and four with PRE above.

In [ ]:
rows = []
for key, g in EXC.groupby(["inc", "kernel", "norm", "behav"]):
    g = g[g.n > 0]
    sig, up, down = 0, 0, 0
    for _, r in g.iterrows():
        if len(r.PRE) > 1 and stats.mannwhitneyu(r.PRE, r.POST).pvalue < 0.01:
            sig += 1; up += np.median(r.PRE) < np.median(r.POST); down += np.median(r.PRE) > np.median(r.POST)
    rows.append(dict(inclusion=key[0], kernel=key[1], norm=key[2], behav=key[3], sessions=len(g), significant=sig,
                     fraction=sig / max(len(g), 1), PRE_below_POST=up, PRE_above_POST=down))
F4S = pd.DataFrame(rows)
print(F4S.to_string(float_format=lambda v: f"{v:.3f}"))
print("Paper: 53 sessions (tolerance 5.3), fraction 0.132 (tolerance 0.0132), both directions")

**Result: fig4-sessions-few-change not reproduced.** 21 to 23 sessions have included pairs (paper 53 +- 5.3). At most 1 session per reading has PRE and POST different at p < 0.01 (fraction 0 to 0.048, against 0.132), and no reading has both directions. Under the whole-epoch inclusion variant (not in the claims file, asked of a reader), 54 sessions have pairs, within tolerance, and 0 to 10 sessions differ at p < 0.01 (fraction up to 0.185); none has PRE above POST, so both directions still never occur.

## fig4c-temporal-bias
> The temporal ordering of LS spike pairs during HFOs before and after the maze correlates with the ordering during traversal to a
> similar degree (P = 0.85).

B = CCG summed over +1..+200 ms minus over -200..-1 ms (the zero bin left out). Per recording, the correlation over pairs of
B(PRE) with B(BEHAV) and of B(BEHAV) with B(POST); recordings with at least 3 pairs. Two-sided t-test, paired by recording and two-sample.

In [ ]:
rows = []
for bm, lst in PAIRS.items():
    for inc in ["a", "b"]:
        for nm in ["raw", "sqrt"]:
            for cor in ["pearson", "spearman"]:
                r1, r2 = [], []
                for s, P in lst:
                    m = P[f"inc_{inc}"]
                    if m.sum() < 3: continue
                    B = {}
                    for ep in ["PRE", "BEHAV", "POST"]:
                        c = P["ccg"][ep][m]
                        if nm == "sqrt": c = c / P["norm"][ep][m][:, None]
                        B[ep] = c[:, 201:].sum(1) - c[:, :200].sum(1)
                    f = stats.pearsonr if cor == "pearson" else stats.spearmanr
                    a, b = f(B["PRE"], B["BEHAV"])[0], f(B["BEHAV"], B["POST"])[0]
                    if np.isfinite(a) and np.isfinite(b): r1.append(a); r2.append(b)
                r1, r2 = np.array(r1), np.array(r2)
                rows.append(dict(behav=bm, inclusion=inc, norm=nm, corr=cor, recordings=len(r1), PRE_BEHAV=r1.mean(), BEHAV_POST=r2.mean(),
                                 diff=(r2 - r1).mean(), p_paired=stats.ttest_rel(r1, r2).pvalue, p_two_sample=stats.ttest_ind(r1, r2).pvalue))
F4C = pd.DataFrame(rows)
print(F4C.to_string(float_format=lambda v: f"{v:.3g}"))
print(f"Readings where the t-test finds a difference at p < 0.05: paired {int((F4C.p_paired < .05).sum())}, two-sample {int((F4C.p_two_sample < .05).sum())} of {len(F4C)}")

**Result: fig4c-temporal-bias reproduced.** No reading finds the PRE-BEHAV and BEHAV-POST correlations different (paired p 0.060 to 0.90, two-sample 0.13 to 0.91; mean BEHAV-POST minus PRE-BEHAV -0.017 to 0.194), over 16 to 19 recordings with at least 3 pairs, against the paper's 53.

## fig4d-compression
> Cross-correlograms of LS pairs during HFOs correlate with those during traversal above chance up to about 20-fold temporal
> compression (P < 0.01), similarly for PRE and POST (P > 0.01 for all compression factors).

BEHAV CCG over +-120 ms (241 points); the HFO CCG over +-120/c ms, linearly interpolated to 241 points, for c = 1..50; Pearson
correlation. Chance: the same correlation with the BEHAV CCG circularly shifted by a random amount (one shift per pair and factor,
the claim's seed). Tests per factor over pairs: actual against chance by signed-rank and by rank-sum (actual higher, p < 0.01);
PRE against POST by rank-sum.

In [ ]:
FACT = np.arange(1, 51)
LAGS_MS = np.arange(-200, 201)
WARP = np.stack([np.stack([np.interp(np.linspace(-120 / c, 120 / c, 241), LAGS_MS, np.eye(401)[k]) for k in range(401)], 1) for c in FACT])  # [50, 241, 401]
def rowcorr(X, Y):
    X = X - X.mean(-1, keepdims=True); Y = Y - Y.mean(-1, keepdims=True)
    return (X * Y).sum(-1) / np.sqrt((X ** 2).sum(-1) * (Y ** 2).sum(-1))
def compression(pairs_list, inc, seed, hfo_eps=("PRE", "POST")):
    rng = np.random.default_rng(seed)
    out = {ep: [] for ep in hfo_eps}; ch = {ep: [] for ep in hfo_eps}
    for s, P in pairs_list:
        m = P[f"inc_{inc}"]
        if m.sum() == 0: continue
        beh = P["ccg"]["BEHAV"][m][:, 80:321]                          # +-120 ms
        for ep in hfo_eps:
            h = P["ccg"][ep][m]
            Wd = np.einsum("fkl,pl->pfk", WARP, h)                     # pairs x factors x 241
            out[ep].append(rowcorr(Wd, beh[:, None, :]))
            shift = rng.integers(1, 241, size=Wd.shape[:2])
            idx = (np.arange(241)[None, None, :] + shift[..., None]) % 241
            bsh = np.take_along_axis(np.broadcast_to(beh[:, None, :], Wd.shape), idx, axis=-1)
            ch[ep].append(rowcorr(Wd, bsh))
    return {ep: np.vstack(v) for ep, v in out.items()}, {ep: np.vstack(v) for ep, v in ch.items()}
def compression_summary(act, cha, sig=0.01):
    res = {}
    for ep in act:
        p_sr, p_rs = [], []
        for f in range(len(FACT)):
            a, c = act[ep][:, f], cha[ep][:, f]; ok = np.isfinite(a) & np.isfinite(c)
            p_sr.append(stats.wilcoxon(a[ok], c[ok], alternative="greater").pvalue if ok.sum() > 5 else 1)
            p_rs.append(stats.mannwhitneyu(a[ok], c[ok], alternative="greater").pvalue if ok.sum() > 5 else 1)
        p_sr, p_rs = np.array(p_sr), np.array(p_rs)
        def largest(p):
            bad = np.flatnonzero(p >= sig); return int(FACT[bad[0] - 1]) if len(bad) and bad[0] > 0 else (int(FACT[-1]) if not len(bad) else 0)
        res[ep] = dict(p_sr=p_sr, p_rs=p_rs, largest_sr=largest(p_sr), largest_rs=largest(p_rs))
    pp = np.array([stats.mannwhitneyu(act["PRE"][:, f][np.isfinite(act["PRE"][:, f])], act["POST"][:, f][np.isfinite(act["POST"][:, f])]).pvalue for f in range(len(FACT))])
    res["PRE_POST_p"] = pp
    return res
CMP = {}
for bm in ["trials", "span"]:
    for inc in ["a", "b"]:
        act, cha = compression(PAIRS[bm], inc, seed_of("fig4d-compression"))
        r = compression_summary(act, cha); CMP[(bm, inc)] = (act, cha, r)
        print(f"behav {bm}, inclusion {inc}: {len(act['PRE'])} pairs; largest factor with every factor up to it above chance (p < 0.01): "
              f"PRE signed-rank {r['PRE']['largest_sr']}, rank-sum {r['PRE']['largest_rs']}; POST signed-rank {r['POST']['largest_sr']}, rank-sum {r['POST']['largest_rs']}; "
              f"factors with PRE != POST at p < 0.01: {int((r['PRE_POST_p'] < 0.01).sum())} (smallest p {r['PRE_POST_p'].min():.2g})")
        print(f"   median correlation at c = 1, 5, 10, 20, 50: PRE {[round(float(np.nanmedian(act['PRE'][:, f-1])), 3) for f in (1, 5, 10, 20, 50)]}, "
              f"chance {[round(float(np.nanmedian(cha['PRE'][:, f-1])), 3) for f in (1, 5, 10, 20, 50)]}; "
              f"POST {[round(float(np.nanmedian(act['POST'][:, f-1])), 3) for f in (1, 5, 10, 20, 50)]}")
act, cha, r = CMP[("trials", "a")]
fig, ax = plt.subplots(figsize=(5, 3))
for ep in ["PRE", "POST"]:
    ax.plot(FACT, np.nanmean(act[ep], 0), label=ep); ax.plot(FACT, np.nanmean(cha[ep], 0), "--", label=f"{ep} chance")
ax.set_xlabel("compression factor"); ax.set_ylabel("mean r with BEHAV CCG"); ax.legend(fontsize=7); plt.show()
print(f"Plotted range {min(np.nanmean(act[e], 0).min() for e in act):.3f} to {max(np.nanmean(act[e], 0).max() for e in act):.3f}")

In [ ]:
# The circular shift is random: the largest factor under four more seeds (behav trials, inclusion a).
for k in range(1, 5):
    sd = seed_of("fig4d-compression") + 1000 * k
    act_k, cha_k = compression(PAIRS["trials"], "a", sd); r = compression_summary(act_k, cha_k)
    print(f"seed {sd}: largest factor PRE signed-rank {r['PRE']['largest_sr']}, rank-sum {r['PRE']['largest_rs']}; POST {r['POST']['largest_sr']}, {r['POST']['largest_rs']}")

**Result: fig4d-compression partially reproduced.** PRE and POST CCGs correlate with BEHAV above chance at factor 1 in every reading. Every factor up to 5 to 8 (PRE) and 8 to 11 (POST) is above chance at p < 0.01 under the claim's seed (signed-rank and rank-sum), against the paper's ~20 (18 to 22). Under four other seeds the largest factor is 5 to 12. PRE and POST do not differ at p < 0.01 at any factor (smallest p 0.055).

## figs3d-hpc-compression-post
> For hippocampal cell pairs, compression of behavioural spike patterns during CA1 ripples is higher after than before the
> behavioural session, for compression factors 1 to 7; HPC pairs are the only ones with a PRE to POST difference.

As fig4d-compression, for pairs of hippocampal units of the same region label (`hpc` with `hpc`, `ca3` with `ca3`), with spikes
inside CA1Ripples events in PRE and POST and BEHAV spikes in trials. Readings: pairs across shanks and all pairs; inclusion (a) and (b).
POST against PRE by rank-sum over pairs at each factor.

In [ ]:
S3_SESS = [s for s in S if s.has_trials and ca1_ev(s) is not None and len(s.units_in(HPC)) >= 2]
print(f"Recordings with trials, CA1Ripples and two or more hippocampal units: {len(S3_SESS)} ({collections.Counter(s.subject for s in S3_SESS)})")
t0 = time.time()
HP = {}
for cross in [True, False]:
    lst = []
    for s in S3_SESS:
        for reg in ["hpc", "ca3", "ca1"]:
            us = s.units_in([reg])
            if len(us) >= 2: lst.append((s, session_pairs(s, us, ca1_ev, "trials", cross_shank=cross)))
    HP[cross] = lst
print(f"CCGs in {time.time()-t0:.0f} s")
for cross in [True, False]:
    for inc in ["a", "b"]:
        act, cha = compression(HP[cross], inc, seed_of("figs3d-hpc-compression-post"))
        pp = np.array([stats.mannwhitneyu(act["POST"][:, f][np.isfinite(act["POST"][:, f])], act["PRE"][:, f][np.isfinite(act["PRE"][:, f])]).pvalue for f in range(7)])
        hi = np.array([np.nanmedian(act["POST"][:, f]) > np.nanmedian(act["PRE"][:, f]) for f in range(7)])
        print(f"{'across shanks' if cross else 'all pairs'}, inclusion {inc}: {len(act['PRE'])} pairs; factors 1-7 with POST > PRE at p < 0.01: {int(((pp < .01) & hi).sum())}; "
              f"p = {np.round(pp, 4).tolist()}; median r PRE {np.round(np.nanmedian(act['PRE'][:, :7], 0), 3).tolist()}, POST {np.round(np.nanmedian(act['POST'][:, :7], 0), 3).tolist()}")

**Result: figs3d-hpc-compression-post partially reproduced.** Over 9366 to 15066 hippocampal pairs in 42 recordings, the POST correlation with BEHAV exceeds the PRE one at p < 0.01 at 1 to 3 of the factors 1 to 7 (factor 3 in every reading, p <= 0.0012), not at most of them; POST medians are at or above PRE medians at every factor from 1 to 7.

## figs2-post-rate-increase
> Firing rates increase during post-behavior NREM sleep in CA1, CA3 and LS.

Per neuron, the rate in NREM within PRE, in BEHAV (trials) and in NREM within POST, divided by the neuron's rate over the whole
session (Figure S2B legend, change relative to the session average). Recordings with trials and NREM of at least 60 s in both PRE
and POST. CA1 is read as units labelled `hpc` or `ca1` (the dataset labels CA1 units `hpc`), CA3 as `ca3`; as a variant, `hpc`
units in recordings with a `ca3_channel` flag are counted as CA3. Two-sided t-tests of POST against PRE, paired and two-sample.

In [ ]:
rows = []
for s in S:
    if not s.has_trials or s.sleep is None: continue
    ep = s.epochs("trials"); nrem = s.state_intervals("NREM")
    pre_n, post_n = T.intersect(nrem, ep["PRE"]), T.intersect(nrem, ep["POST"])
    if T.total(pre_n) < 60 or T.total(post_n) < 60: continue
    ca3flag = bool(s.electrodes.ca3_channel.any())
    for u, reg in enumerate(s.unit_region):
        sp = s.spikes[u]; avg = len(sp) / s.duration
        if avg == 0: continue
        r = dict(session=s.session, subject=s.subject, region=reg, region_flag=("ca3" if (reg == "hpc" and ca3flag) else reg))
        for nm, iv in [("PRE", pre_n), ("BEHAV", ep["BEHAV"]), ("POST", post_n)]:
            r[nm] = T.in_intervals(sp, iv).sum() / T.total(iv) / avg
        rows.append(r)
FR = pd.DataFrame(rows)
FR["label"] = FR.region.map({"ls": "LS", "hpc": "CA1", "ca1": "CA1", "ca3": "CA3"})
FR["label_flag"] = FR.region_flag.map({"ls": "LS", "hpc": "CA1", "ca1": "CA1", "ca3": "CA3"})
print(f"Recordings: {FR.session.nunique()}")
for col in ["label", "label_flag"]:
    print(f"Regions by {'unit label' if col == 'label' else 'unit label and ca3_channel flag'}:")
    for reg in ["LS", "CA1", "CA3"]:
        g = FR[FR[col] == reg]
        if len(g) == 0: print(f"  {reg}: none"); continue
        tp, ti = stats.ttest_rel(g.POST, g.PRE), stats.ttest_ind(g.POST, g.PRE)
        print(f"  {reg}: {len(g)} neurons in {g.session.nunique()} recordings, {g.subject.nunique()} animals; relative rate PRE NREM {g.PRE.mean():.3f}, "
              f"BEHAV {g.BEHAV.mean():.3f}, POST NREM {g.POST.mean():.3f}; POST > PRE in {(g.POST > g.PRE).mean():.2f}; paired p = {tp.pvalue:.2g} (t = {tp.statistic:.2f}), "
              f"two-sample p = {ti.pvalue:.2g} (t = {ti.statistic:.2f})")
print("Paper N: LS 1754, CA1 2315, CA3 1373 (tolerance 10%)")
print("\nPer animal, LS, mean POST - PRE relative rate:", FR[FR.label == "LS"].assign(d=lambda x: x.POST - x.PRE).groupby("subject").d.agg(["mean", "count"]).round(3).to_dict("index"))

**Result: figs2-post-rate-increase partially reproduced.** Relative to the session average, LS rates are higher in POST NREM than in PRE NREM (1.334 against 1.265; paired p = 0.032, two-sample p = 0.036; 1570 neurons). CA1 rates (units labelled `hpc`) are lower in POST NREM (1.293 against 1.347; paired p = 0.043), and CA3 rates do not change (p = 0.59). The ca3_channel reading gives the same conclusions (CA1 lower, p = 0.018; CA3 unchanged, p = 0.94). Neuron counts are 1570, 2597 and 466 against 1754, 2315 and 1373.

## figs2c-no-ls-reactivation
> After controlling for the change in firing rate, by rate matching or by bin size, LS-LS pairs show no fast timescale reactivation
> beyond chance, and HPC and HPC-LS pairs give similar results.

EV and REV (Kudrimoti et al. 1999; Pennartz et al. 2004, STAR Methods formula) from pair correlation vectors of binned counts in PRE NREM
ripple epochs, BEHAV (trials) and POST NREM ripple epochs. Ripple epochs are read as (a) CA1Ripples events in NREM (start to stop) and
(b) for LS pairs, LSRipples events in NREM. Counts in 25 ms and 1 s bins over the concatenated epoch time. Rate matching (Giri et al.
2019): single spikes removed at random from the epoch (PRE or POST NREM) where the neuron fires faster, until the rates match; 10
repetitions averaged (the claim's seed). Recordings with trials, NREM in PRE and POST, and at least 5 neurons of the set (pairs within
a recording). Two-sided t-test of EV against REV over recordings, paired and two-sample.

In [ ]:
def binned(spikes, iv, bs):
    """Counts in bins of bs over the concatenated intervals: each spike's time is mapped onto the intervals laid end to end."""
    iv = T.merge_intervals(iv)
    off = np.r_[0, np.cumsum(iv[:, 1] - iv[:, 0])]
    nb = max(int(off[-1] // bs), 1)
    out = []
    for sp in spikes:
        k = np.searchsorted(iv[:, 0], sp, side="right") - 1
        ok = (k >= 0) & (sp <= iv[np.maximum(k, 0), 1])
        tc = sp[ok] - iv[k[ok], 0] + off[k[ok]]
        out.append(np.bincount(np.minimum((tc // bs).astype(int), nb - 1), minlength=nb)[:nb])
    return np.vstack(out)
def corrvec(X, pairs):
    C = np.corrcoef(X); v = C[pairs[0], pairs[1]]
    return v
def ev_rev(pre, beh, post):
    ok = np.isfinite(pre) & np.isfinite(beh) & np.isfinite(post)
    if ok.sum() < 3: return np.nan, np.nan
    r = lambda a, b: np.corrcoef(a[ok], b[ok])[0, 1]
    rbp, rbq, rpq = r(beh, post), r(beh, pre), r(pre, post)
    ev = ((rbp - rbq * rpq) / np.sqrt((1 - rbq ** 2) * (1 - rpq ** 2))) ** 2
    rev = ((rbq - rbp * rpq) / np.sqrt((1 - rbp ** 2) * (1 - rpq ** 2))) ** 2
    return ev, rev
def rate_match(sp_pre, sp_post, T_pre, T_post, rng):
    a, b = [], []
    for x, y in zip(sp_pre, sp_post):
        ra, rb = len(x) / T_pre, len(y) / T_post
        if ra > rb: x = np.sort(rng.choice(x, int(round(rb * T_pre)), replace=False))
        elif rb > ra: y = np.sort(rng.choice(y, int(round(ra * T_post)), replace=False))
        a.append(x); b.append(y)
    return a, b
def reactivation(seed, n_rep=10, bins=(0.025, 1.0)):
    rng = np.random.default_rng(seed)
    out = []
    for s in S:
        if not s.has_trials or s.sleep is None: continue
        ep = s.epochs("trials"); nrem = s.state_intervals("NREM")
        for evname, evfn in [("CA1 ripples", ca1_ev), ("LS HFOs", ls_ev)]:
            e = evfn(s)
            if e is None: continue
            rip = e[["start_time", "stop_time"]].to_numpy()
            pre_iv, post_iv = T.intersect(T.intersect(rip, nrem), ep["PRE"]), T.intersect(T.intersect(rip, nrem), ep["POST"])
            if T.total(pre_iv) < 1 or T.total(post_iv) < 1: continue
            ls, hp = s.units_in(["ls"]), s.units_in(HPC)
            sets = {"LS-LS": (ls, ls), "HPC-HPC": (hp, hp), "HPC-LS": (hp, ls)}
            if evname == "LS HFOs": sets = {"LS-LS": (ls, ls)}
            for nm, (ua, ub) in sets.items():
                units = np.unique(np.r_[ua, ub])
                if len(ua) < 5 or len(ub) < 5: continue
                pos = {u: k for k, u in enumerate(units)}
                if nm in ("LS-LS", "HPC-HPC"):
                    I, J = np.triu_indices(len(ua), 1); pairs = (np.array([pos[u] for u in ua[I]]), np.array([pos[u] for u in ua[J]]))
                else:
                    I, J = np.meshgrid(np.arange(len(ua)), np.arange(len(ub)), indexing="ij")
                    pairs = (np.array([pos[u] for u in ua[I.ravel()]]), np.array([pos[u] for u in ub[J.ravel()]]))
                sp = [s.spikes[u] for u in units]
                spre, spost = T.restrict(sp, pre_iv), T.restrict(sp, post_iv)
                for bs in bins:
                    beh = corrvec(binned(sp, ep["BEHAV"], bs), pairs)
                    raw = ev_rev(corrvec(binned(spre, pre_iv, bs), pairs), beh, corrvec(binned(spost, post_iv, bs), pairs))
                    evs, revs = [], []
                    for k in range(n_rep):
                        a, b = rate_match(spre, spost, T.total(pre_iv), T.total(post_iv), rng)
                        e_, r_ = ev_rev(corrvec(binned(a, pre_iv, bs), pairs), beh, corrvec(binned(b, post_iv, bs), pairs))
                        evs.append(e_); revs.append(r_)
                    out.append(dict(session=s.session, subject=s.subject, events=evname, set=nm, bin=bs, EV=raw[0], REV=raw[1],
                                    EV_rm=np.nanmean(evs), REV_rm=np.nanmean(revs)))
    return pd.DataFrame(out)
t0 = time.time()
RE = reactivation(seed_of("figs2c-no-ls-reactivation"))
print(f"computed in {time.time()-t0:.0f} s")
def re_table(RE):
    rows = []
    for (evn, st, bs), g in RE.groupby(["events", "set", "bin"]):
        for m in ["", "_rm"]:
            d = g.dropna(subset=[f"EV{m}", f"REV{m}"])
            if len(d) < 2: continue
            rows.append(dict(events=evn, set=st, bin=bs, rate_matched=bool(m), recordings=len(d), EV=d[f"EV{m}"].mean(), REV=d[f"REV{m}"].mean(),
                             p_paired=stats.ttest_rel(d[f"EV{m}"], d[f"REV{m}"]).pvalue, p_two_sample=stats.ttest_ind(d[f"EV{m}"], d[f"REV{m}"]).pvalue))
    D = pd.DataFrame(rows); D["EV>REV p<0.05 (paired)"] = (D.p_paired < .05) & (D.EV > D.REV); D["EV>REV p<0.05 (two-sample)"] = (D.p_two_sample < .05) & (D.EV > D.REV)
    return D
RET = re_table(RE)
print(RET.to_string(float_format=lambda v: f"{v:.3g}"))

In [ ]:
# Rate matching is random: the 25 ms, rate-matched rows under four more seeds.
for k in range(1, 5):
    sd = seed_of("figs2c-no-ls-reactivation") + 1000 * k
    D = re_table(reactivation(sd, bins=(0.025,))); D = D[(D.bin == 0.025) & D.rate_matched]
    print(f"seed {sd}:", "; ".join(f"{r.events} {r.set} EV {r.EV:.3g} REV {r.REV:.3g} p {r.p_paired:.2g}/{r.p_two_sample:.2g}" for r in D.itertuples()))

**Result: figs2c-no-ls-reactivation reproduced.** With 25 ms bins after rate matching, EV does not exceed REV for LS-LS pairs (CA1 ripple epochs, paired p = 0.23; LS HFO epochs, p = 0.57), HPC-HPC pairs (p = 0.18) or HPC-LS pairs (p = 0.33), and the same holds under four other seeds. The paper's increase of EV over REV with 1 s bins appears only for HPC-HPC pairs without rate matching (paired p = 0.049), so the positive control is weak here and the absence for LS-LS pairs has little power behind it.

## Summary

| Claim | Outcome | Notebook |
|---|---|---|
| dataset-composition | partially reproduced (4 of 6 counts, assumed: `hpc` units counted as CA1 and ripplesALL as CA1 ripples; CA3 and LS HFO-rate recordings short) | main, LFP |
| fig1c-hfo-each-animal | partially reproduced (4 of 6 animals at the event bin, 5 of 6 at +-5 ms) | main |
| fig1c-post-burst-suppression | not reproduced (assumed window from the event stop) | main |
| fig1c-event-count | reproduced (27014 against 29955) | main |
| figs1-participation | not reproduced (5.7% to 10.7% of neurons, 0.079 to 0.155 spikes) | main |
| fig1d-phase-locking | partially reproduced (61% to 81% locked; trough preference holds) | LFP, own-shank |
| fig1f-shorter-than-ripples | reproduced | main |
| fig1g-slower-than-ripples | reproduced | main |
| fig1h-no-amplitude-frequency-correlation | reproduced (assumed: deposited maximum NSS; the recomputed NSS correlates weakly) | main, LFP |
| fig2-confined-to-ls | not reproduced | LFP |
| fig2ef-intermediate-ls | blocked: metadata_insufficient | |
| fig3-travelling-waves | blocked: metadata_insufficient | |
| fig4b-pre-post-unchanged | partially reproduced (assumed normalized CCG; 361 to 503 pairs against 7172; a whole-epoch inclusion variant, asked of a reader, keeps 10159 to 15732 pairs and finds a small difference in 4 of 8 normalized readings) | main |
| fig4b-hfo-above-behav | reproduced (assumed: fails only with raw counts and BEHAV over the whole span) | main |
| fig4-sessions-few-change | not reproduced | main |
| fig4c-temporal-bias | reproduced | main |
| fig4d-compression | partially reproduced (largest factor 5 to 11 against ~20) | main |
| figs3d-hpc-compression-post | partially reproduced (1 to 3 of factors 1-7) | main |
| figs2-post-rate-increase | partially reproduced (LS only) | main |
| figs2c-no-ls-reactivation | reproduced | main |
| fig5b-nrem-and-rates | reproduced | main |
| fig5d-lag | reproduced (6.2 to 6.4 ms) | main, LFP |
| fig5e-state-coupling | partially reproduced (assumed pooled HFOs; not reproduced over recordings) | main |
| fig5f-ripple-triggered-firing | not reproduced | main |
| fig5f-no-phase-coherence | not reproduced | LFP |
| ls-spikes-not-locked-to-ca1-ripples | not reproduced (assumed reading inside ripples; 7.2% against 5%) | LFP |
| fig6-optogenetic-hfos | blocked: data_missing | |
| fig7e-ca1-spikes, fig7e-state, fig7e-duration | partially reproduced (claims file's t-test: far above the paper's percentages, chance 22% to 44%; the z reading, asked of a reader, is within tolerance in some combinations) | Figure 7E |
| fig7e-ca1-depth | blocked: metadata_insufficient | |
| fig7e-replay, fig7e-seqnmf, figs5-seqnmf-variance | blocked: other (not attempted within the run's time) | |
| fig7e-content-independent, fig7e-coupled-only | blocked: other (content features not attempted) | |
| discussion-medial-septum | not reproduced | main, LFP |
| discussion-cingulate-down-state | not reproduced | main, LFP |
| fig8-model | blocked: other (not attempted within the run's time) | |

A failure to reproduce here may be a failure of this attempt, not of the paper: see the reasoning and the differences from the paper's method.

How the analysis differs from the paper's: The lab's deposited detections are used for LS HFOs (LSRipples) and CA1 ripples (CA1Ripples); their stored parameters differ from the Methods (LS: 120-180 Hz, durations up to 100 ms, peak thresholds of 4 to 6 SD in places; CA1: 130-200 Hz). PRE, BEHAV and POST are not marked in the files and are taken as before the first trial, from the first trial to the last, and after the last (an assumption). Where the paper leaves a method open, every reading the claims file names was run; the choices adopted as assumptions are the units labelled `hpc` as CA1 and ripplesALL as CA1 ripples where CA1Ripples is missing (dataset-composition), the normalized CCG (Figure 4), the deposited maximum NSS as the amplitude (Figure 1H), the window from the event stop (post-burst suppression), spikes inside CA1 ripples (locking to CA1) and the fraction over pooled HFOs (Figure 5E inset). The Figure 7E test is the claims file's t-test of the shuffled MSEs against the actual MSE, whose chance rate is 22% to 44%; the z reading of the figure's axis is shown beside it and asked of a reader. CA1 and CA3 are told apart by the units' region label, which marks most hippocampal units only `hpc`. Figure 2 site ratios use two 14-minute chunks of each recording. The Figure 7E Poisson fits were run under the claim's seed only.

What was found about the dataset: Six rats (DT1, DT2, DT5, DT7, DT8, DT9) with 2551 LS units in 82 recordings and 98 files; no recording is deposited twice. The two optogenetic rats are not in the dandiset. Electrode positions and the depth of each recording within LS are not stored, nor are the PRE, BEHAV and POST periods. Most hippocampal units are labelled `hpc`, so CA1 and CA3 populations cannot be told apart reliably (10 recordings have units labelled `ca3`, 19 have a `ca3_channel` flag). 36 units of DT7 20170407 have an empty region label. In 3 files, less than half of the trial time is scored WAKE (DT2 3540um-1288um 20160227, 12.5%; DT5 20161014, 20.3%; DT8 20170201, 31.6%). The DT9 CA3 recordings have no CA1 ripple table although the paper detected SPW-Rs on a CA1 reference electrode for CA3 recordings. The deposited LS detections use parameters that differ from the Methods, as the files' descriptions say. The electrode and units tables hold no site positions; in the one file per animal listed, the units of DT1, DT2 and DT8 hold one 32-sample waveform each (`raw_waveform`) and those of DT5, DT7 and DT9 none. On the septal shank of the DT7 recordings, the region labels of the sites (ls, ms, acg, cc) alternate along the channel order, so the labels cannot be checked against the sites' geometry.
